# 07 · Modelos de sentimiento

**Objetivo:** comparar cuatro formas de conseguir un modelo que mida el sentimiento de un texto financiero: **M1**, el diccionario Loughran-McDonald (sin entrenar nada); **M2**, un Transformer entrenado solo con las frases etiquetadas de **Financial PhraseBank**; **M3**, la misma red preentrenada como BERT con el corpus propio; y **M4**, FinBERT, un BERT financiero preentrenado por otros (añadido después del examen, sección 11). Todos usan la misma partición y las mismas métricas. M2 y M3 siguen el código de los apuntes de clase (`Transformers.ipynb` y `bert_simplificado.ipynb`).

**Plan:**

| Paso | Qué se hace | Estado |
|---|---|---|
| 1 | Financial PhraseBank: lectura, clases, longitudes, repetidas y solapamiento con el corpus | hecho |
| 2 | Partición estratificada 60/20/20 y referencia "siempre neutral" | hecho |
| 3 | Función de evaluación: F1 macro con intervalo por bootstrap, exactitud y matriz de confusión | hecho |
| 4 | **M1**, diccionario Loughran-McDonald; umbral de "neutral" elegido en validación | hecho |
| 5 | Vocabulario común de M2 y M3 (`TextVectorization` con `[mask]`) | hecho |
| 6 | **M2**, Transformer; épocas y pesos de clase elegidos en validación | hecho |
| 7 | **M3**, preentrenamiento adivinando palabras tapadas, 2 épocas | hecho |
| 8 | **M3**, ajuste con PhraseBank: congelado y después descongelado | hecho |
| 9 | Examen con el test, una sola vez; sensibilidad con AllAgree | hecho |
| 10 | Corrección del preentrenamiento de M3 (encontrada después del examen) | hecho |
| 11 | **M4**, FinBERT (añadido después del examen) | hecho |

**Entradas:**
- Financial PhraseBank (Malo et al., 2014), descargado en `data/phrasebank/`
- `data/cache/corpus_preentrenamiento.parquet` (notebook 06): texto sin etiquetas para preentrenar M3
- FinBERT preentrenado (`yiyanghkust/finbert-pretrain` en Hugging Face, licencia Apache-2.0)

**Salidas:**
- `data/cache/phrasebank_particion.parquet`: las frases de 50Agree sin repetidas, con su parte (entrenamiento, validación o test) y si están en AllAgree
- `data/modelos/`: `vocabulario.txt`; `m1_diccionario.json`; `m2_sin_pesos.keras` y `m2_con_pesos.keras` con sus historiales y `m2_eleccion.json`; `m3_preentrenado.keras` y su historial
- `data/modelos/`: `m3_congelado.keras` y `m3_descongelado.keras` (M3 ajustado) con sus historiales; `m3_preentrenado_mezclado.keras`, `m3_mezclado_congelado.keras` y `m3_mezclado_descongelado.keras` (M3 corregido, el oficial) con sus historiales
- `data/modelos/`: `finbert_pretrain/` (FinBERT descargado y convertido) y `m4_finbert.weights.h5` (M4 ajustado) con su historial
- `data/cache/predicciones_phrasebank.parquet`: lo que dice cada modelo de cada frase de la validación y el test

In [1]:
import hashlib
import json
import os
import re
import shutil
import time
import warnings
import zipfile
from pathlib import Path

os.environ["KERAS_BACKEND"] = "tensorflow"   # motor de Keras del proyecto; se fija antes de importar keras
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"       # sin los mensajes internos de TensorFlow (no afectan a los resultados)

import keras
import keras_hub
import numpy as np
import pandas as pd
import tensorflow as tf
from keras import layers
from keras.layers import TextVectorization
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
import plotly.express as px
from dotenv import load_dotenv
from huggingface_hub import hf_hub_download

from funciones import (
    CARPETA_CACHE,
    CLASES_SENTIMIENTO,
    SEMILLA,
    clasificar_tono,
    comparar_f1,
    comprobar_descarga_manual,
    evaluar_clasificacion,
    tono_loughran_mcdonald,
)

tf.get_logger().setLevel("ERROR")                # ni sus avisos de rendimiento
load_dotenv()
print("Keras", keras.__version__, "· motor:", keras.backend.backend())

Keras 3.15.1 · motor: tensorflow


## 1. ¿Con qué frases etiquetadas aprenden los modelos? (Financial PhraseBank)

**Financial PhraseBank** (Malo et al., 2014, [arXiv 1307.5336](https://arxiv.org/abs/1307.5336)): unas 4.800 frases de noticias financieras en inglés, cada una etiquetada como **negativa, neutral o positiva**. Licencia CC BY-NC-SA 3.0 (uso académico, citando el artículo). Se descargó a mano el 21 oct 2026 de [Hugging Face](https://huggingface.co/datasets/takala/financial_phrasebank) (`FinancialPhraseBank-v1.0.zip`), y el notebook comprueba su huella SHA-256 antes de usarlo: si fuera otra versión, se para.

In [2]:
CARPETA_PHRASEBANK = CARPETA_CACHE.parent / "phrasebank"
RUTA_ZIP = CARPETA_PHRASEBANK / "FinancialPhraseBank-v1.0.zip"
comprobar_descarga_manual(RUTA_ZIP,
                          sha256_esperado="0e1a06c4900fdae46091d031068601e3773ba067c7cecb5b0da1dcba5ce989a6",
                          enlace_descarga="https://huggingface.co/datasets/takala/financial_phrasebank/resolve/main/data/FinancialPhraseBank-v1.0.zip")

with zipfile.ZipFile(RUTA_ZIP) as comprimido:
    ficheros = [nombre for nombre in comprimido.namelist() if nombre.startswith("FinancialPhraseBank-v1.0/") and nombre.endswith(".txt")]
    comprimido.extractall(CARPETA_PHRASEBANK, members=ficheros)
print("Ficheros:", [nombre.split("/")[1] for nombre in ficheros])

Ficheros: ['License.txt', 'README.txt', 'Sentences_50Agree.txt', 'Sentences_66Agree.txt', 'Sentences_75Agree.txt', 'Sentences_AllAgree.txt']


### 1.1 Los ficheros por dentro

Una frase por línea, con la etiqueta tras la última `@`, en codificación **latin-1** (leído como UTF-8, falla). Las frases traen espacios antes de la puntuación (*"Gran , the company"*), como las dejaron los autores.

In [3]:
RUTA_50AGREE = CARPETA_PHRASEBANK / "FinancialPhraseBank-v1.0" / "Sentences_50Agree.txt"
for linea in RUTA_50AGREE.read_text(encoding="latin-1").splitlines()[:3]:
    print(linea)


def leer_phrasebank(variante: str) -> pd.DataFrame:
    """Lee una variante de PhraseBank ('50Agree', 'AllAgree'…) y devuelve una tabla con frase y etiqueta."""
    ruta = CARPETA_PHRASEBANK / "FinancialPhraseBank-v1.0" / f"Sentences_{variante}.txt"
    filas = []
    for linea in ruta.read_text(encoding="latin-1").splitlines():
        frase, etiqueta = linea.rsplit("@", 1)
        filas.append({"frase": frase.strip(), "etiqueta": etiqueta.strip()})
    return pd.DataFrame(filas)


frases_50 = leer_phrasebank("50Agree")
frases_todas = leer_phrasebank("AllAgree")
print(f"50Agree: {len(frases_50):,} frases · AllAgree: {len(frases_todas):,} frases")

According to Gran , the company has no plans to move all production to Russia , although that is where the company is growing .@neutral
Technopolis plans to develop in stages an area of no less than 100,000 square meters in order to host companies working in computer technologies and telecommunications , the statement said .@neutral
The international electronic industry company Elcoteq has laid off tens of employees from its Tallinn facility ; contrary to earlier layoffs the company contracted the ranks of its office workers , the daily Postimees reported .@negative
50Agree: 4,846 frases · AllAgree: 2,264 frases


### 1.2 ¿Cuántas frases hay de cada clase?

In [4]:
distribucion = pd.DataFrame({
    "50Agree": frases_50["etiqueta"].value_counts(),
    "50Agree (%)": (frases_50["etiqueta"].value_counts(normalize=True) * 100).round(1),
    "AllAgree": frases_todas["etiqueta"].value_counts(),
    "AllAgree (%)": (frases_todas["etiqueta"].value_counts(normalize=True) * 100).round(1),
})
distribucion

,50Agree,50Agree (%),AllAgree,AllAgree (%)
etiqueta,,,,
neutral,2879,59.4,1391,61.4
positive,1363,28.1,570,25.2
negative,604,12.5,303,13.4


### 1.2 Longitud, ejemplos y frases repetidas

Longitud en palabras (separando por espacios), dos ejemplos de cada clase y las frases repetidas. Una frase repetida con etiquetas **distintas** es un problema: el modelo recibe dos respuestas contradictorias para la misma pregunta.

In [5]:
frases_50["palabras"] = frases_50["frase"].str.split().str.len()
print(frases_50.groupby("etiqueta")["palabras"].describe(percentiles=[0.5]).round(0)[["count", "min", "50%", "max"]], "\n")

for nombre, tabla in [("50Agree", frases_50), ("AllAgree", frases_todas)]:
    repetidas = tabla[tabla.duplicated(subset="frase", keep=False)]
    etiquetas_por_frase = repetidas.groupby("frase")["etiqueta"].nunique()
    print(f"{nombre}: {repetidas['frase'].nunique()} frases repetidas ({len(repetidas)} filas)"
          f" · con etiquetas distintas: {(etiquetas_por_frase > 1).sum()}")

pd.set_option("display.max_colwidth", 160)
display(frases_50.groupby("etiqueta").sample(2, random_state=SEMILLA)[["etiqueta", "frase"]])
con_conflicto = frases_50.groupby("frase")["etiqueta"].nunique()
frases_50[frases_50["frase"].isin(con_conflicto[con_conflicto > 1].index)].sort_values("frase")

           count  min   50%   max
etiqueta                         
negative   604.0  5.0  22.0  56.0
neutral   2879.0  2.0  21.0  81.0
positive  1363.0  5.0  23.0  57.0 

50Agree: 8 frases repetidas (16 filas) · con etiquetas distintas: 2
AllAgree: 5 frases repetidas (10 filas) · con etiquetas distintas: 0


,etiqueta,frase
3790,negative,The company decided at the end of 2008 to temporarily shut down its ammonia plant in Billingham and extend the maintenance period at its Ince facility .
4670,negative,down to EUR5 .9 m H1 '09 3 August 2009 - Finnish media group Ilkka-Yhtyma Oyj ( HEL : ILK2S ) said today its net profit fell 45 % on the year to EUR5 .9 m i...
1636,neutral,The mill is concentrating on getting the supercalendered line running satisfactorily before restarting its older newsprint line .
1117,neutral,Wellness services will be important in the future .
2310,positive,The fund at fair value will increase correspondingly .
2317,positive,This new partnership agreement represents a significant milestone for both parties .


,frase,etiqueta,palabras
78,TELECOMWORLDWIRE-7 April 2006-TJ Group Plc sells stake in Morning Digital Design Oy Finnish IT company TJ Group Plc said on Friday 7 April that it had signe...,neutral,42
79,TELECOMWORLDWIRE-7 April 2006-TJ Group Plc sells stake in Morning Digital Design Oy Finnish IT company TJ Group Plc said on Friday 7 April that it had signe...,positive,42
788,The Group 's business is balanced by its broad portfolio of sports and presence in all major markets .,positive,19
789,The Group 's business is balanced by its broad portfolio of sports and presence in all major markets .,neutral,19


### 1.3 ¿Aparece alguna frase de PhraseBank en nuestro corpus?

Si una frase del test estuviera en el corpus de preentrenamiento (notebook 06), M3 la habría leído antes de examinarse. Se comparan en minúsculas y **sin espacios**, porque PhraseBank pone espacios antes de la puntuación y el corpus no.

In [6]:
def normalizar(texto: str) -> str:
    """Texto en minúsculas y sin espacios, para comparar frases escritas con distinto espaciado."""
    return re.sub(r"\s+", "", texto.lower())


corpus = pd.read_parquet(CARPETA_CACHE / "corpus_preentrenamiento.parquet", columns=["texto", "fuente"])
corpus_normalizado = set(corpus["texto"].map(normalizar))
en_el_corpus = frases_50["frase"].map(normalizar).isin(corpus_normalizado)
print(f"Frases de PhraseBank (50Agree) que aparecen en el corpus: {en_el_corpus.sum()} de {len(frases_50):,}")

Frases de PhraseBank (50Agree) que aparecen en el corpus: 0 de 4,846


**Conclusión:**

- **4.846 frases en 50Agree y 2.264 en AllAgree**, desequilibradas: el 59 % son **neutrales** y solo 1 de cada 8 **negativa**. Decir siempre "neutral" acertaría el 59 % sin aprender nada; por eso la métrica principal es el **F1 macro**, que da el mismo peso a las tres clases.
- Frases **cortas**: 22 palabras de mediana y 81 como mucho.
- **8 frases repetidas en 50Agree, 2 de ellas con etiquetas contradictorias.** En AllAgree, 5 repetidas, todas con la misma etiqueta.
- **Ninguna frase de PhraseBank está en el corpus:** el preentrenamiento no ha visto el test. Basta con buscar frases idénticas, porque PhraseBank son noticias finlandesas de 2004–2011 y el corpus empieza en 2016.

## 2. ¿Con qué frases se entrena, se ajusta y se examina cada modelo? (partición)

Antes de partir se quitan las repetidas: las **2 contradictorias**, del todo (no se sabe cuál es la etiqueta buena), y de las demás se deja una sola copia. Si no, una misma frase podría caer en el entrenamiento y en el test.

In [7]:
etiquetas_por_frase = frases_50.groupby("frase")["etiqueta"].nunique()
contradictorias = etiquetas_por_frase[etiquetas_por_frase > 1].index

frases = frases_50[~frases_50["frase"].isin(contradictorias)].drop_duplicates(subset="frase").reset_index(drop=True)
frases["en_allagree"] = frases["frase"].isin(frases_todas["frase"])
print(f"50Agree: {len(frases_50):,} filas → sin contradictorias ni repetidas: {len(frases):,} frases"
      f" ({len(frases_50) - len(frases)} filas fuera) · en AllAgree: {frases['en_allagree'].sum():,}")

50Agree: 4,846 filas → sin contradictorias ni repetidas: 4,836 frases (10 filas fuera) · en AllAgree: 2,259


### 2.1 Partición estratificada 60 / 20 / 20

Al azar con la semilla, pero con la **misma proporción de clases** en cada parte:

| Parte | Para qué se usa |
|---|---|
| **Entrenamiento** (60 %) | El modelo aprende con estas frases |
| **Validación** (20 %) | Elegir hiperparámetros y cuándo parar de entrenar |
| **Test** (20 %) | Examen final: **se mira una sola vez, con todos los modelos fijados (paso 9)** |

**Diferencia con los apuntes:** allí el test se usa como validación. Es una fuga de información: si se eligen las épocas mirando el test, deja de ser un examen limpio.

In [8]:
resto, test = train_test_split(frases, test_size=0.2, stratify=frases["etiqueta"], random_state=SEMILLA)
entrenamiento, validacion = train_test_split(resto, test_size=0.25, stratify=resto["etiqueta"], random_state=SEMILLA)

particion = pd.concat([entrenamiento.assign(parte="entrenamiento"), validacion.assign(parte="validacion"),
                       test.assign(parte="test")], ignore_index=True)[["frase", "etiqueta", "parte", "en_allagree"]]
particion.to_parquet(CARPETA_CACHE / "phrasebank_particion.parquet", index=False)
print("Frases por parte:", particion["parte"].value_counts().to_dict())

Frases por parte: {'entrenamiento': 2901, 'test': 968, 'validacion': 967}


La proporción de clases de cada parte, y cuántas frases del test están en AllAgree (todos los anotadores de acuerdo), para la medida de sensibilidad:

In [9]:
proporciones = (pd.crosstab(particion["parte"], particion["etiqueta"], normalize="index") * 100).round(1)
proporciones["frases"] = particion["parte"].value_counts()
display(proporciones.loc[["entrenamiento", "validacion", "test"]])

test_allagree = particion[(particion["parte"] == "test") & particion["en_allagree"]]
print(f"Test: {(particion['parte'] == 'test').sum()} frases, de las que {len(test_allagree)} están en AllAgree."
      f" Se medirán aparte como sensibilidad: el mismo examen, solo con las frases en las que todos los anotadores coincidieron.")

etiqueta,negative,neutral,positive,frases
parte,,,,
entrenamiento,12.5,59.4,28.2,2901
validacion,12.5,59.4,28.1,967
test,12.5,59.4,28.1,968


Test: 968 frases, de las que 448 están en AllAgree. Se medirán aparte como sensibilidad: el mismo examen, solo con las frases en las que todos los anotadores coincidieron.


### 2.2 La referencia mínima: decir siempre "neutral"

Cualquier modelo tiene que superarla. Se calcula en la validación, para no mirar el test hasta el final.

In [10]:
reales = particion.loc[particion["parte"] == "validacion", "etiqueta"]
siempre_neutral = ["neutral"] * len(reales)
print(f"Siempre neutral (validación) · exactitud: {accuracy_score(reales, siempre_neutral):.3f}"
      f" · F1 macro: {f1_score(reales, siempre_neutral, average='macro'):.3f}")

Siempre neutral (validación) · exactitud: 0.594 · F1 macro: 0.248


La exactitud sale alta (0,594) y el F1 macro muy bajo (0,248): acertar todas las neutrales ya es el 59 %, pero en las negativas y las positivas este "modelo" saca un 0.

**Conclusión:** 4.836 frases repartidas 60/20/20 con la misma proporción de clases, guardadas para que todos los modelos usen exactamente las mismas. El listón mínimo es un F1 macro de 0,25.

## 3. ¿Cómo se mide cada modelo? (función de evaluación)

Todos los modelos se miden con la misma función, `evaluar_clasificacion` (`funciones.py`; también la usa el notebook 08):

| Medida | Qué es |
|---|---|
| **F1 macro** (principal) | El F1 de cada clase, promediado con el mismo peso para las tres: las negativas cuentan lo mismo que las neutrales |
| **Intervalo de confianza del 95 %** | Por **bootstrap**: 1.000 muestras al azar con reemplazamiento de las mismas frases, el F1 en cada una y los percentiles 2,5 y 97,5 |
| **Exactitud** | Porcentaje de frases acertadas |
| **Matriz de confusión** | Cuántas frases de cada clase real acaban en cada clase predicha |

Se prueba con "siempre neutral", que tiene que dar 0,248 y 0,594:

In [11]:
evaluacion_neutral = evaluar_clasificacion(reales, siempre_neutral)

print(f"Siempre neutral (validación, {evaluacion_neutral['frases']} frases)")
print(f"F1 macro: {evaluacion_neutral['f1_macro']:.3f}  (IC 95 %: {evaluacion_neutral['f1_ic_inferior']:.3f} – {evaluacion_neutral['f1_ic_superior']:.3f})")
print(f"Exactitud: {evaluacion_neutral['exactitud']:.3f}")
evaluacion_neutral["matriz_confusion"]

Siempre neutral (validación, 967 frases)
F1 macro: 0.248  (IC 95 %: 0.240 – 0.257)
Exactitud: 0.594


predicha,negative,neutral,positive
real,,,
negative,0,121,0
neutral,0,574,0
positive,0,272,0


**Conclusión:** la función reproduce las cifras de la referencia. Su intervalo es estrecho (0,24 – 0,26) porque este "modelo" no puede acertar ni fallar de formas distintas; en un modelo de verdad será más ancho, sobre todo por las negativas (unas 120 en la validación).

## 4. M1: ¿cuánto se consigue sin entrenar nada? (diccionario Loughran-McDonald)

El **diccionario Loughran-McDonald** (Loughran y McDonald, 2011, *Journal of Finance*) es la lista de palabras de sentimiento más usada en finanzas. Se construyó con informes 10-K, porque las listas generales fallan en este lenguaje: *liability* o *tax* no son negativas en un informe financiero. Se descargó a mano el 1 nov 2026 de la página de los autores ([sraf.nd.edu](https://sraf.nd.edu/loughranmcdonald-master-dictionary/), versión de marzo de 2026). Licencia: libre para investigación académica, citando el artículo. El notebook comprueba su huella SHA-256.

In [12]:
RUTA_LM = CARPETA_CACHE.parent / "loughran_mcdonald" / "Loughran-McDonald_MasterDictionary_1993-2025.csv"
comprobar_descarga_manual(RUTA_LM,
                          sha256_esperado="e2d1328682bab7d2187684fb9f5420bb730401c9eefc00daf835edd203f4859d",
                          enlace_descarga="https://sraf.nd.edu/loughranmcdonald-master-dictionary/ (enlace CSV)")
print(f"{RUTA_LM.name}: {RUTA_LM.stat().st_size / 1e6:.1f} MB")

Loughran-McDonald_MasterDictionary_1993-2025.csv: 9.1 MB


### 4.1 El fichero por dentro

Una fila por palabra, en mayúsculas, y una columna por cada una de sus siete listas. Cada columna guarda **el año en que la palabra entró** en la lista (0 si nunca estuvo; con signo menos, el año en que se quitó), así que una palabra está en una lista si el valor es **mayor que 0**. Se lee con `keep_default_na=False` para que pandas no convierta en vacía la palabra *NULL*.

M1 usa solo las listas **negativa y positiva**: las otras cinco (incertidumbre, litigio, modales y restricción) no corresponden a ninguna clase de PhraseBank.

In [13]:
diccionario_lm = pd.read_csv(RUTA_LM, keep_default_na=False)
print("Filas:", len(diccionario_lm), "· columnas:", list(diccionario_lm.columns))
display(diccionario_lm.head(3))

LISTAS_LM = ["Negative", "Positive", "Uncertainty", "Litigious", "Strong_Modal", "Weak_Modal", "Constraining"]
display(pd.DataFrame({
    "en vigor (> 0)": (diccionario_lm[LISTAS_LM] > 0).sum(),
    "quitadas (< 0)": (diccionario_lm[LISTAS_LM] < 0).sum(),
}))

palabras_negativas = set(diccionario_lm.loc[diccionario_lm["Negative"] > 0, "Word"])
palabras_positivas = set(diccionario_lm.loc[diccionario_lm["Positive"] > 0, "Word"])
print(f"Negativas: {len(palabras_negativas)} · positivas: {len(palabras_positivas)} · en las dos: {len(palabras_negativas & palabras_positivas)}")
print("Negativas, al azar:", sorted(pd.Series(sorted(palabras_negativas)).sample(12, random_state=SEMILLA)))
print("Positivas, al azar:", sorted(pd.Series(sorted(palabras_positivas)).sample(12, random_state=SEMILLA)))

Filas: 86553 · columnas: ['Word', 'Seq_num', 'Word Count', 'Word Proportion', 'Average Proportion', 'Std Dev', 'Doc Count', 'Negative', 'Positive', 'Uncertainty', 'Litigious', 'Strong_Modal', 'Weak_Modal', 'Constraining', 'Complexity', 'Syllables', 'Source']


,Word,Seq_num,Word Count,Word Proportion,Average Proportion,Std Dev,Doc Count,Negative,Positive,Uncertainty,Litigious,Strong_Modal,Weak_Modal,Constraining,Complexity,Syllables,Source
0,AARDVARK,1,814,3.085383e-08,2.022181e-08,4.059339e-06,158,0,0,0,0,0,0,0,0,2,12of12inf
1,AARDVARKS,2,3,1.137119e-10,7.898767e-12,8.829342e-09,1,0,0,0,0,0,0,0,0,2,12of12inf
2,ABACI,3,9,3.411357e-10,1.067549e-10,5.054031e-08,7,0,0,0,0,0,0,0,0,3,12of12inf


,en vigor (> 0),quitadas (< 0)
Negative,2345,10
Positive,347,7
Uncertainty,297,0
Litigious,903,2
Strong_Modal,19,0
Weak_Modal,27,0
Constraining,184,0


Negativas: 2345 · positivas: 347 · en las dos: 0
Negativas, al azar: ['ANOMALIES', 'CONVICT', 'DISAVOWED', 'DISTRACTS', 'GRATUITOUS', 'GROSSLY', 'INVESTIGATIONS', 'MALICE', 'OVERTURNS', 'RETALIATION', 'STRAIN', 'TOLERATING']
Positivas, al azar: ['CONCLUSIVELY', 'EMPOWERING', 'ENHANCE', 'EXCLUSIVES', 'IMPRESSES', 'INNOVATION', 'LOYAL', 'PROFICIENCY', 'PROSPEROUS', 'STRENGTH', 'SUCCESSFUL', 'SURPASS']


**Conclusión:** 86.553 palabras, de las que 2.345 son negativas y 347 positivas (casi 7 por cada positiva: los 10-K hablan más de riesgos que de logros), y ninguna está en las dos listas. Dos límites: se hizo con informes 10-K, no con noticias, y cuenta palabras sueltas sin ver las negaciones (*not profitable* suma una positiva).

### 4.2 La regla de M1: del recuento de palabras a una clase

$$\text{tono} = \frac{\text{palabras positivas} - \text{palabras negativas}}{\text{palabras de la frase}}$$

Las palabras son los trozos de solo letras de la frase, en mayúsculas. La clase es *positiva* si el tono supera un **umbral**, *negativa* si baja de −umbral y *neutral* si no. Las dos funciones (`tono_loughran_mcdonald` y `clasificar_tono`) están en `funciones.py`, porque el notebook 08 las usa con los titulares.

Unos ejemplos de la validación y, para cada clase real, cuántas frases tienen alguna palabra con tono (sin ninguna, M1 solo puede decir "neutral"):

In [14]:
validacion = particion[particion["parte"] == "validacion"].copy()
validacion["tono_m1"] = [tono_loughran_mcdonald(frase, palabras_negativas, palabras_positivas) for frase in validacion["frase"]]

pd.set_option("display.max_colwidth", 120)
display(pd.concat([validacion.nlargest(2, "tono_m1"), validacion.nsmallest(2, "tono_m1")])[["frase", "etiqueta", "tono_m1"]].round(3))

cobertura = validacion.groupby("etiqueta")["tono_m1"].agg(
    tono_positivo=lambda tono: (tono > 0).mean(),
    tono_negativo=lambda tono: (tono < 0).mean(),
    sin_tono=lambda tono: (tono == 0).mean(),
)
(cobertura * 100).round(1)

,frase,etiqueta,tono_m1
2985,"`` Our decision and resolve on MeeGo is only stronger , '' she continued .",positive,0.182
3533,Finnair was able to operate most of its leisure flights despite the strike .,positive,0.154
3393,Terms were not disclosed .,neutral,-0.250
3585,No financial details were disclosed .,neutral,-0.200


,tono_positivo,tono_negativo,sin_tono
etiqueta,,,
negative,1.7,43.0,55.4
neutral,7.1,8.0,84.8
positive,24.6,7.4,68.0


**El umbral se elige en la validación**: de 0 a 0,20 en pasos de 0,01, el de mayor F1 macro (si empatan, el más pequeño). Es la única decisión de M1 que se toma con datos.

In [15]:
umbrales = np.round(np.arange(0, 0.201, 0.01), 2)
f1_por_umbral = pd.Series(
    [f1_score(validacion["etiqueta"], [clasificar_tono(tono, umbral) for tono in validacion["tono_m1"]], average="macro")
     for umbral in umbrales],
    index=umbrales, name="F1 macro")
umbral_m1 = f1_por_umbral.idxmax()   # idxmax devuelve el primero si hay empate: el umbral más pequeño

grafico = px.line(f1_por_umbral.reset_index(), x="index", y="F1 macro", markers=True,
                  labels={"index": "Umbral de neutral"}, title="M1: F1 macro en la validación según el umbral")
grafico.add_hline(y=0.248, line_dash="dash", annotation_text="siempre neutral")
grafico.show(renderer="notebook_connected")
print(f"Umbral elegido: {umbral_m1} · F1 macro en validación: {f1_por_umbral.max():.3f}")

Umbral elegido: 0.0 · F1 macro en validación: 0.509


Los umbrales de 0 a 0,02 dan lo mismo (en una frase de 20–30 palabras, una sola palabra de tono ya da ±0,03), y a partir de ahí el F1 cae. El mejor es **0**. M1 con ese umbral:

In [16]:
validacion["prediccion_m1"] = [clasificar_tono(tono, umbral_m1) for tono in validacion["tono_m1"]]
evaluacion_m1 = evaluar_clasificacion(validacion["etiqueta"], validacion["prediccion_m1"])

print(f"M1 (validación, {evaluacion_m1['frases']} frases)")
print(f"F1 macro: {evaluacion_m1['f1_macro']:.3f}  (IC 95 %: {evaluacion_m1['f1_ic_inferior']:.3f} – {evaluacion_m1['f1_ic_superior']:.3f})")
print(f"Exactitud: {evaluacion_m1['exactitud']:.3f}")
evaluacion_m1["matriz_confusion"]

M1 (validación, 967 frases)
F1 macro: 0.509  (IC 95 %: 0.472 – 0.545)
Exactitud: 0.627


predicha,negative,neutral,positive
real,,,
negative,52,67,2
neutral,46,487,41
positive,20,185,67


Se guardan el umbral y las dos listas, para que el notebook 08 aplique exactamente el mismo M1:

In [17]:
RUTA_M1 = CARPETA_CACHE.parent / "modelos" / "m1_diccionario.json"
RUTA_M1.parent.mkdir(parents=True, exist_ok=True)
RUTA_M1.write_text(json.dumps({
    "umbral": float(umbral_m1),
    "palabras_negativas": sorted(palabras_negativas),
    "palabras_positivas": sorted(palabras_positivas),
}, indent=1), encoding="utf-8")
print(f"Guardado: {RUTA_M1.name} - {RUTA_M1.stat().st_size / 1e3:.0f} KB")

Guardado: m1_diccionario.json - 42 KB


**Conclusión:** sin entrenar nada, M1 llega a un **F1 macro de 0,51** en la validación (IC 95 %: 0,47 – 0,55), el doble que decir siempre "neutral". Su límite es la **cobertura**: el 76 % de las frases no tiene ninguna palabra del diccionario, y M1 las llama neutrales. Falla sobre todo en las **positivas** (185 de 272 acaban como neutrales): la lista positiva es corta y está pensada para informes 10-K. Es el listón para M2 y M3.

## 5. ¿Con qué palabras trabajan M2 y M3? (vocabulario común)

Una red lee números, no texto. Como en `bert_simplificado.ipynb`, la capa `TextVectorization` estandariza el texto (minúsculas, sin puntuación), aprende las `VOCAB_SIZE` palabras más frecuentes (las demás pasan a `[UNK]`), añade al final el token `[mask]` que usará M3, y corta o rellena cada frase a `MAX_LEN` posiciones. M2 y M3 comparten vocabulario, aprendido solo con el corpus y el **entrenamiento** de PhraseBank.

**Diferencias con los apuntes:**
- `"[mask]"` en minúscula y `special_tokens` definido: con `"[MASK]"`, la estandarización lo convierte en `[UNK]` y se taparían palabras con "desconocida" (error de los apuntes).
- `MAX_LEN = 64` en lugar de 256: son frases, no críticas de cine.
- El vocabulario se aprende sin el test (los apuntes usan el test de IMDB).
- `keras.layers` (Keras 3) y vocabulario guardado en disco.

In [18]:
MAX_LEN = 64          # palabras por frase (se corta o se rellena)
VOCAB_SIZE = 30000    # tamaño del vocabulario, como en los apuntes


def custom_standardization(input_data):
    """Minúsculas, sin <br /> y sin signos de puntuación (como en los apuntes)."""
    lowercase = tf.strings.lower(input_data)
    stripped_html = tf.strings.regex_replace(lowercase, "<br />", " ")
    return tf.strings.regex_replace(
        stripped_html, "[%s]" % re.escape("!#$%&'()*+,-./:;<=>?@\\^_`{|}~"), ""
    )


def get_vectorize_layer(texts, vocab_size, max_seq, special_tokens=["[mask]"]):
    """Crea la capa TextVectorization, aprende el vocabulario con `texts` y añade los tokens especiales al final (como en los apuntes)."""
    vectorize_layer = TextVectorization(
        max_tokens=vocab_size,
        output_mode="int",
        standardize=custom_standardization,
        output_sequence_length=max_seq,
    )
    vectorize_layer.adapt(texts)

    # Insert mask token in vocabulary
    vocab = vectorize_layer.get_vocabulary()
    vocab = vocab[2 : vocab_size - len(special_tokens)] + special_tokens
    vectorize_layer.set_vocabulary(vocab)
    return vectorize_layer

Se aprende una vez y se guarda en `data/modelos/vocabulario.txt`; al volver a ejecutar se lee de disco, así M2, M3 y el notebook 08 usan los mismos números.

In [19]:
corpus = pd.read_parquet(CARPETA_CACHE / "corpus_preentrenamiento.parquet")["texto"]
frases_entrenamiento = particion.loc[particion["parte"] == "entrenamiento", "frase"]
RUTA_VOCABULARIO = CARPETA_CACHE.parent / "modelos" / "vocabulario.txt"

if RUTA_VOCABULARIO.exists():
    vectorize_layer = TextVectorization(max_tokens=VOCAB_SIZE, output_mode="int",
                                        standardize=custom_standardization, output_sequence_length=MAX_LEN)
    vectorize_layer.set_vocabulary(RUTA_VOCABULARIO.read_text(encoding="utf-8").splitlines())
    print("Vocabulario leído de disco")
else:
    vectorize_layer = get_vectorize_layer(corpus.tolist() + frases_entrenamiento.tolist(), VOCAB_SIZE, MAX_LEN)
    RUTA_VOCABULARIO.write_text("\n".join(vectorize_layer.get_vocabulary()[2:]), encoding="utf-8")
    print("Vocabulario aprendido y guardado")

# Número del token [mask], para el preentrenamiento de M3 (como en los apuntes)
mask_token_id = int(vectorize_layer(["[mask]"]).numpy()[0][0])
vocabulario = [str(palabra) for palabra in vectorize_layer.get_vocabulary()]
print(f"Palabras en el vocabulario: {len(vocabulario)} · primeras: {vocabulario[:12]} · [mask] = {mask_token_id}")

Vocabulario leído de disco
Palabras en el vocabulario: 30000 · primeras: ['', '[UNK]', 'the', 'of', 'and', 'to', 'in', 'our', 'for', 'million', 'a', 'on'] · [mask] = 29999


Así queda una frase de la validación (las palabras poco frecuentes, como nombres de empresa, salen como `[UNK]`):

In [20]:
frase_ejemplo = validacion["frase"].iloc[0]
numeros = vectorize_layer([frase_ejemplo]).numpy()[0]
print(frase_ejemplo, "\n")
print("Números:", numeros[numeros != 0].tolist(), f"(+ {(numeros == 0).sum()} ceros de relleno)\n")
print("Palabras:", [vocabulario[numero] for numero in numeros if numero != 0])

20 October 2010 - Finnish environmental management company Lassila & Tikanoja Oyj HEL : LAT1V , or L&T , said Monday it expects its operating profit , excluding non-recurring items , for the whole 2010 to be slightly lower than in 2009 . 

Números: [348, 290, 2581, 3941, 870, 140, 82, 1, 1, 5958, 9206, 1, 15, 12666, 1523, 2305, 171, 892, 105, 48, 221, 395, 2330, 408, 8, 2, 2522, 2581, 5, 47, 1245, 98, 170, 6, 3441] (+ 29 ceros de relleno)

Palabras: ['20', 'october', '2010', 'finnish', 'environmental', 'management', 'company', '[UNK]', '[UNK]', 'oyj', 'hel', '[UNK]', 'or', 'lt', 'said', 'monday', 'it', 'expects', 'its', 'operating', 'profit', 'excluding', 'nonrecurring', 'items', 'for', 'the', 'whole', '2010', 'to', 'be', 'slightly', 'lower', 'than', 'in', '2009']


**¿Basta con 64 palabras?** ¿Y cuántas palabras de la validación no están en el vocabulario?

In [21]:
def contar_palabras(textos: list[str]) -> np.ndarray:
    """Palabras de cada texto después de estandarizarlo, sin cortar a MAX_LEN."""
    return tf.strings.split(custom_standardization(textos)).row_lengths().numpy()


longitudes = {"PhraseBank": contar_palabras(particion["frase"].tolist()), "Corpus": contar_palabras(corpus.tolist())}
display(pd.DataFrame({
    nombre: {"mediana": np.median(n), "percentil 95": np.percentile(n, 95), "máximo": n.max(),
             f"% con más de {MAX_LEN}": round(100 * (n > MAX_LEN).mean(), 1)}
    for nombre, n in longitudes.items()
}).T)

numeros_validacion = vectorize_layer(validacion["frase"].tolist()).numpy()
palabras_validacion = numeros_validacion[numeros_validacion != 0]
print(f"Palabras [UNK]: {(palabras_validacion == 1).mean():.1%} de las palabras de la validación")
print(f"Frases con alguna [UNK]: {(numeros_validacion == 1).any(axis=1).mean():.1%}")

,mediana,percentil 95,máximo,% con más de 64
PhraseBank,19.0,37.0,52.0,0.0
Corpus,19.0,60.0,112.0,3.9


Palabras [UNK]: 6.4% de las palabras de la validación
Frases con alguna [UNK]: 60.7%


**Conclusión:** vocabulario de 30.000 palabras con `[mask]` en la última posición. Con 64 posiciones no se corta **ninguna** frase de PhraseBank (la más larga tiene 52 palabras) y solo el 3,9 % de las del corpus. El 6,4 % de las palabras de la validación queda como `[UNK]`, sobre todo nombres de empresas y códigos. Al quitar la puntuación, *non-recurring* pasa a ser *nonrecurring*: es el precio de una estandarización sencilla como la de clase.

## 6. M2: ¿cuánto aprende un Transformer solo con las frases etiquetadas?

M2 aprende desde cero, solo con las 2.901 frases de entrenamiento. Las capas son las de `Transformers.ipynb`: **`PositionalEmbedding`** (un vector por palabra más otro por su posición) y **`TransformerEncoder`** (atención de varias cabezas y una red densa, con conexiones residuales y normalización; la máscara hace que la atención ignore el relleno). La parte final es la de `bert_simplificado.ipynb`: `GlobalMaxPooling1D`, `Dense(64, relu)` y la salida de 3 clases. **M3 tendrá exactamente la misma red.**

**Diferencias con los apuntes:**
- **3 clases:** `Dense(3, activation="softmax")` y `sparse_categorical_crossentropy`.
- Parte final, Adam y tamaño (128 valores por palabra, 8 cabezas, 128 neuronas) de `bert_simplificado`, para que M2 y M3 solo se diferencien en el preentrenamiento.
- `compute_mask` con `keras.ops.not_equal` (Keras 3).
- **Épocas con `EarlyStopping` sobre la validación**, no el test (F. Chollet, *Deep Learning with Python*, 2.ª ed., cap. 7).
- Pesos de clase probados y semilla fija con determinismo.

In [22]:
EMBED_DIM = 128   # valores por palabra (como en bert_simplificado)
NUM_HEAD = 8      # cabezas de atención
FF_DIM = 128      # neuronas de la red densa del encoder
BATCH_SIZE = 32

# Determinismo: con la misma semilla, TensorFlow da exactamente los mismos resultados en cada ejecución (algo más despacio)
tf.config.experimental.enable_op_determinism()

# Keras avisa de que TransformerEncoder "destruye la máscara": la usa dentro de la atención (llega a call como
# argumento `mask`), pero no la pasa a la capa siguiente. Es lo mismo que en los apuntes; se explica en 6.1.
warnings.filterwarnings("ignore", message=".*does not support masking.*")


class TransformerEncoder(layers.Layer):
    def __init__(self, embed_dim, dense_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.dense_dim = dense_dim
        self.num_heads = num_heads
        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim)
        self.dense_proj = keras.Sequential(
            [layers.Dense(dense_dim, activation="relu"),
             layers.Dense(embed_dim),]
        )
        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()

    def call(self, inputs, mask=None):  # Aquí se construye el Transformer
        if mask is not None:
            mask = mask[:, tf.newaxis, :]
        attention_output = self.attention(
            inputs, inputs, attention_mask=mask)
        proj_input = self.layernorm_1(inputs + attention_output)
        proj_output = self.dense_proj(proj_input)
        return self.layernorm_2(proj_input + proj_output)

    # sirve para serializar, lo que hace posible hacer save y load del modelo
    def get_config(self):
        config = super().get_config()
        config.update({
            "embed_dim": self.embed_dim,
            "num_heads": self.num_heads,
            "dense_dim": self.dense_dim,
        })
        return config


class PositionalEmbedding(layers.Layer):
    def __init__(self, sequence_length, input_dim, output_dim, **kwargs):
        super().__init__(**kwargs)
        self.token_embeddings = layers.Embedding(
            input_dim=input_dim, output_dim=output_dim)
        self.position_embeddings = layers.Embedding(
            input_dim=sequence_length, output_dim=output_dim)
        self.sequence_length = sequence_length
        self.input_dim = input_dim
        self.output_dim = output_dim

    def call(self, inputs):
        length = tf.shape(inputs)[-1]
        positions = tf.range(start=0, limit=length, delta=1)
        embedded_tokens = self.token_embeddings(inputs)
        embedded_positions = self.position_embeddings(positions)
        return embedded_tokens + embedded_positions

    def compute_mask(self, inputs, mask=None):
        return keras.ops.not_equal(inputs, 0)   # Diferencia con los apuntes: tf.math.not_equal no vale en Keras 3

    def get_config(self):
        config = super().get_config()
        config.update({
            "output_dim": self.output_dim,
            "sequence_length": self.sequence_length,
            "input_dim": self.input_dim,
        })
        return config


CAPAS_PROPIAS = {"TransformerEncoder": TransformerEncoder, "PositionalEmbedding": PositionalEmbedding}   # para cargar los .keras

### 6.1 Datos y red

Cada frase pasa a sus 64 números y cada etiqueta a 0, 1 o 2 (negativa, neutral, positiva). La red se construye con una función para empezar siempre desde cero con la misma semilla. El encoder no pasa la máscara a la capa siguiente, así que `GlobalMaxPooling1D` también ve el relleno, igual que en los apuntes (el aviso de Keras se oculta).

In [23]:
def preparar(parte: str) -> tuple[np.ndarray, np.ndarray]:
    """Números de las frases de una parte de la partición y sus etiquetas (0, 1, 2)."""
    tabla = particion[particion["parte"] == parte]
    x = vectorize_layer(tabla["frase"].tolist()).numpy()
    y = tabla["etiqueta"].map({clase: numero for numero, clase in enumerate(CLASES_SENTIMIENTO)}).to_numpy()
    return x, y


x_entrenamiento, y_entrenamiento = preparar("entrenamiento")
x_validacion, y_validacion = preparar("validacion")
print("Entrenamiento:", x_entrenamiento.shape, "· validación:", x_validacion.shape)

Entrenamiento: (2901, 64) · validación: (967, 64)


In [24]:
def crear_m2():
    """Transformer encoder (Transformers.ipynb) con la parte final de bert_simplificado, sin preentrenar."""
    keras.utils.set_random_seed(SEMILLA)   # mismos pesos iniciales cada vez
    inputs = keras.Input(shape=(None,), dtype="int64")
    x = PositionalEmbedding(MAX_LEN, VOCAB_SIZE, EMBED_DIM)(inputs)
    x = TransformerEncoder(EMBED_DIM, FF_DIM, NUM_HEAD)(x)
    x = layers.GlobalMaxPooling1D()(x)
    x = layers.Dense(64, activation="relu")(x)
    outputs = layers.Dense(3, activation="softmax")(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer=keras.optimizers.Adam(),
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    return model


crear_m2().summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_embeddi… │ (None, None, 128) │  3,848,192 │ input_layer[0][0] │
│ (PositionalEmbeddi… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, None)      │          0 │ input_layer[0][0] │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_encoder │ (None, None, 128) │    561,024 │ positional_embed… │
│ (TransformerEncode… │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 128)       │          0 │ transformer_enco… │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 64)        │      8,256 │ global_max_pooli… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 3)         │        195 │ dense_2[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 4,417,667 (16.85 MB)

 Trainable params: 4,417,667 (16.85 MB)

 Non-trainable params: 0 (0.00 B)

### 6.2 Cuántas épocas: `EarlyStopping`

Se entrena hasta `MAX_EPOCAS`. Si la pérdida de validación no mejora en `PACIENCIA` épocas seguidas, se para, y con `restore_best_weights=True` el modelo se queda con los pesos de la **mejor época**. `entrenar` guarda el modelo y su historial en `data/modelos/`: al volver a ejecutar se leen de disco y no se entrena otra vez. Sirve también para las dos fases de M3.

In [25]:
MAX_EPOCAS = 50
PACIENCIA = 3
CARPETA_MODELOS = CARPETA_CACHE.parent / "modelos"


def entrenar(crear_modelo, nombre: str, pesos_de_clase: dict | None = None):
    """Entrena con EarlyStopping sobre la pérdida de validación, o lo lee de disco si ya está entrenado.

    Al entrenar se ve la barra de progreso de Keras; al leerlo de disco, un resumen.
    Devuelve el modelo (con los pesos de la mejor época) y su historial (pérdidas por época).
    """
    ruta_modelo = CARPETA_MODELOS / f"{nombre}.keras"
    ruta_historial = CARPETA_MODELOS / f"{nombre}_historial.json"
    if ruta_modelo.exists() and ruta_historial.exists():
        historial = json.loads(ruta_historial.read_text(encoding="utf-8"))
        print(f"{nombre}: ya entrenado, se lee de disco ({len(historial['loss'])} épocas,"
              f" la mejor la {int(np.argmin(historial['val_loss'])) + 1})")
        return keras.models.load_model(ruta_modelo, custom_objects=CAPAS_PROPIAS), historial

    model = crear_modelo()
    parada = keras.callbacks.EarlyStopping(monitor="val_loss", patience=PACIENCIA, restore_best_weights=True)
    historia = model.fit(x_entrenamiento, y_entrenamiento, validation_data=(x_validacion, y_validacion),
                         epochs=MAX_EPOCAS, batch_size=BATCH_SIZE, class_weight=pesos_de_clase,
                         callbacks=[parada])
    historial = {clave: [float(valor) for valor in valores] for clave, valores in historia.history.items()}
    model.save(ruta_modelo)
    ruta_historial.write_text(json.dumps(historial), encoding="utf-8")
    print(f"{nombre}: {len(historial['loss'])} épocas, la mejor es la {int(np.argmin(historial['val_loss'])) + 1}")
    return model, historial


def dibujar_perdidas(historial: dict, titulo: str) -> None:
    """Pérdida de entrenamiento y de validación por época, con la época elegida marcada."""
    epocas = list(range(1, len(historial["loss"]) + 1))
    mejor = int(np.argmin(historial["val_loss"])) + 1
    grafico = px.line(pd.DataFrame({"época": epocas, "entrenamiento": historial["loss"], "validación": historial["val_loss"]}),
                      x="época", y=["entrenamiento", "validación"], markers=True,
                      labels={"value": "pérdida", "variable": ""}, title=titulo)
    grafico.add_vline(x=mejor, line_dash="dash", annotation_text=f"época elegida: {mejor}")
    grafico.show(renderer="notebook_connected")

### 6.3 Con y sin pesos de clase

Con el desequilibrio de PhraseBank, una red puede aprender a decir casi siempre "neutral". Los **pesos de clase** (`"balanced"`: frases totales / (3 × frases de la clase)) hacen que equivocarse en una clase poco frecuente cueste más. Se entrena M2 con y sin ellos, con la misma semilla:

In [26]:
pesos = compute_class_weight("balanced", classes=np.arange(3), y=y_entrenamiento)
pesos_de_clase = dict(enumerate(pesos))
print({clase: round(peso, 2) for clase, peso in zip(CLASES_SENTIMIENTO, pesos)})

{'negative': np.float64(2.67), 'neutral': np.float64(0.56), 'positive': np.float64(1.18)}


In [27]:
m2_sin_pesos, historial_sin_pesos = entrenar(crear_m2, "m2_sin_pesos")
m2_con_pesos, historial_con_pesos = entrenar(crear_m2, "m2_con_pesos", pesos_de_clase)

m2_sin_pesos: ya entrenado, se lee de disco (5 épocas, la mejor la 2)
m2_con_pesos: ya entrenado, se lee de disco (4 épocas, la mejor la 1)


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/layer.py:431: UserWarning:

`build()` was called on layer 'positional_embedding_1', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/layer.py:431: UserWarning:

`build()` was called on layer 'transformer_encoder_1', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras

In [28]:
dibujar_perdidas(historial_sin_pesos, "M2 sin pesos de clase: pérdida por época")
dibujar_perdidas(historial_con_pesos, "M2 con pesos de clase: pérdida por época")

Las dos versiones se miden en la validación con la misma función que todos los modelos, y M2 es la que tenga **mayor F1 macro**:

In [29]:
def predecir(model, x: np.ndarray) -> list[str]:
    """Clase con más probabilidad para cada frase."""
    return [CLASES_SENTIMIENTO[numero] for numero in model.predict(x, verbose=0).argmax(axis=1)]


reales_validacion = [CLASES_SENTIMIENTO[numero] for numero in y_validacion]
evaluaciones_m2 = {
    "sin pesos de clase": evaluar_clasificacion(reales_validacion, predecir(m2_sin_pesos, x_validacion)),
    "con pesos de clase": evaluar_clasificacion(reales_validacion, predecir(m2_con_pesos, x_validacion)),
}
pd.DataFrame({
    version: {"F1 macro": e["f1_macro"], "IC 95 % inferior": e["f1_ic_inferior"], "IC 95 % superior": e["f1_ic_superior"],
              "exactitud": e["exactitud"]}
    for version, e in evaluaciones_m2.items()
}).T.round(3)

,F1 macro,IC 95 % inferior,IC 95 % superior,exactitud
sin pesos de clase,0.675,0.639,0.712,0.754
con pesos de clase,0.621,0.582,0.658,0.724


In [30]:
version_m2 = max(evaluaciones_m2, key=lambda version: evaluaciones_m2[version]["f1_macro"])
m2 = m2_con_pesos if version_m2 == "con pesos de clase" else m2_sin_pesos
evaluacion_m2 = evaluaciones_m2[version_m2]
historial_m2 = historial_con_pesos if version_m2 == "con pesos de clase" else historial_sin_pesos
(CARPETA_MODELOS / "m2_eleccion.json").write_text(json.dumps({"version": version_m2}), encoding="utf-8")

print(f"M2 elegido: {version_m2}")
print(f"F1 macro: {evaluacion_m2['f1_macro']:.3f}  (IC 95 %: {evaluacion_m2['f1_ic_inferior']:.3f} – {evaluacion_m2['f1_ic_superior']:.3f})")
print(f"Exactitud: {evaluacion_m2['exactitud']:.3f}")
evaluacion_m2["matriz_confusion"]

M2 elegido: sin pesos de clase
F1 macro: 0.675  (IC 95 %: 0.639 – 0.712)
Exactitud: 0.754


predicha,negative,neutral,positive
real,,,
negative,54,42,25
neutral,8,513,53
positive,11,99,162


**Conclusión:** M2 **sin pesos de clase** llega en la validación a un **F1 macro de 0,675** (IC 95 %: 0,64 – 0,71), frente a 0,51 de M1: los intervalos no se solapan. Aprender de 2.901 frases etiquetadas supera a la lista de palabras de los expertos. La comparación definitiva es la del test, en el paso 9.

- **Se sobreajusta muy pronto.** La mejor época es la 2 (sin pesos) o la 1 (con pesos). Desde ahí, la pérdida de entrenamiento sigue bajando (de 0,37 a 0,07) mientras la de validación sube (de 0,66 a 1,31): la red memoriza las frases de entrenamiento. Tiene 4,4 millones de parámetros para 2.901 frases. Es justo lo que el preentrenamiento de M3 podría mejorar.
- **Los pesos de clase no ayudan aquí:** 0,621 frente a 0,675, aunque los intervalos se tocan (0,58 – 0,66 y 0,64 – 0,71). Se queda la versión sin pesos.
- **Errores:** acierta 54 de 121 negativas, 513 de 574 neutrales y 162 de 272 positivas. Las negativas siguen siendo lo más difícil: 42 acaban como neutrales y 25 como positivas.

## 7. M3, primera fase: preentrenar la red adivinando palabras tapadas

Antes de ver ninguna frase etiquetada, M3 aprende cómo es el lenguaje financiero con el corpus propio (695.136 frases sin etiquetas), como BERT en `bert_simplificado.ipynb`: se tapan palabras y la red tiene que adivinarlas por el contexto. `get_masked_input_and_labels`, copiada de los apuntes, elige al azar el 15 % de las palabras de cada frase; de ellas, el 80 % pasa a `[mask]`, el 10 % a una palabra al azar y el 10 % se queda igual. La pérdida solo cuenta en las elegidas (`sample_weights`).

**Diferencias con los apuntes:**
- **La red es la de M2** en lugar de `bert_module`, para que M2 y M3 sean la misma red (por eso el embedding de posición se aprende y va dentro de una capa).
- **Pérdida con la reducción por defecto** en lugar de `Reduction.NONE`: los `sample_weights` ya anulan las palabras no elegidas.
- Nombres de capa sin `/` (Keras 3), `.keras` en lugar de `.h5` y semilla fija.
- Como en los apuntes, *the* (el número 2) nunca se tapa: la condición `encoded_texts <= 2` se escribió para un vocabulario con tres números especiales.

In [31]:
def get_masked_input_and_labels(encoded_texts):
    """Tapa el 15 % de las palabras como BERT (como en los apuntes). Devuelve frases tapadas, palabras originales y pesos."""
    # 15% BERT masking
    inp_mask = np.random.rand(*encoded_texts.shape) < 0.15
    # Do not mask special tokens
    inp_mask[encoded_texts <= 2] = False
    # Set targets to -1 by default, it means ignore
    labels = -1 * np.ones(encoded_texts.shape, dtype=int)
    # Set labels for masked tokens
    labels[inp_mask] = encoded_texts[inp_mask]

    # Prepare input
    encoded_texts_masked = np.copy(encoded_texts)
    # Set input to [MASK] which is the last token for the 90% of tokens
    # This means leaving 10% unchanged
    inp_mask_2mask = inp_mask & (np.random.rand(*encoded_texts.shape) < 0.90)
    encoded_texts_masked[inp_mask_2mask] = mask_token_id  # mask token is the last in the dict

    # Set 10% to a random token
    inp_mask_2random = inp_mask_2mask & (np.random.rand(*encoded_texts.shape) < 1 / 9)
    encoded_texts_masked[inp_mask_2random] = np.random.randint(
        3, mask_token_id, inp_mask_2random.sum()
    )

    # Prepare sample_weights to pass to .fit() method
    sample_weights = np.ones(labels.shape)
    sample_weights[labels == -1] = 0

    # y_labels would be same as encoded_texts i.e input tokens
    y_labels = np.copy(encoded_texts)

    return encoded_texts_masked, y_labels, sample_weights

Se convierte todo el corpus a números y se tapa una sola vez, con la semilla fija (en los apuntes también se tapa una sola vez, antes de entrenar):

In [32]:
x_all_review = vectorize_layer(corpus.tolist()).numpy()
np.random.seed(SEMILLA)
x_masked_train, y_masked_labels, sample_weights = get_masked_input_and_labels(x_all_review)

tapadas = sample_weights > 0
print(f"Frases: {len(x_all_review):,} · palabras tapadas: {tapadas.sum() / (x_all_review != 0).sum():.1%} de las palabras reales")
print(f"De las tapadas: {(x_masked_train[tapadas] == mask_token_id).mean():.1%} con [mask], "
      f"{(x_masked_train[tapadas] == x_all_review[tapadas]).mean():.1%} sin cambiar y el resto, una palabra al azar")

ejemplo = 0
print("\nOriginal:", " ".join(vocabulario[n] for n in x_all_review[ejemplo] if n != 0))
print("Tapada:  ", " ".join(vocabulario[n] for n in x_masked_train[ejemplo] if n != 0))

Frases: 695,136 · palabras tapadas: 14.1% de las palabras reales
De las tapadas: 80.0% con [mask], 10.0% sin cambiar y el resto, una palabra al azar

Original: 6 stocks that are beating the market
Tapada:   6 stocks that are beating the [mask]


La red de preentrenamiento: la de M2 hasta el encoder y, encima, una capa con una salida por cada palabra del vocabulario (`mlm_cls`), que da la probabilidad de cada palabra en cada posición. Es lo que en los apuntes es `create_masked_language_bert_model`:

In [33]:
def create_masked_language_bert_model():
    """Red de preentrenamiento: PositionalEmbedding + TransformerEncoder (la de M2) + una salida por palabra del vocabulario."""
    keras.utils.set_random_seed(SEMILLA)
    inputs = layers.Input((MAX_LEN,), dtype="int64")
    embeddings = PositionalEmbedding(MAX_LEN, VOCAB_SIZE, EMBED_DIM)(inputs)
    encoder_output = TransformerEncoder(EMBED_DIM, FF_DIM, NUM_HEAD, name="encoder_0")(embeddings)
    mlm_output = layers.Dense(VOCAB_SIZE, name="mlm_cls", activation="softmax")(encoder_output)
    mlm_model = keras.Model(inputs, mlm_output)
    mlm_model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
                      loss=keras.losses.SparseCategoricalCrossentropy())
    return mlm_model


create_masked_language_bert_model().summary()

Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_4       │ (None, 64)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_embeddi… │ (None, 64, 128)   │  3,848,192 │ input_layer_4[0]… │
│ (PositionalEmbeddi… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_3         │ (None, 64)        │          0 │ input_layer_4[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_0           │ (None, 64, 128)   │    561,024 │ positional_embed… │
│ (TransformerEncode… │                   │            │ not_equal_3[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mlm_cls (Dense)     │ (None, 64, 30000) │  3,870,000 │ encoder_0[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 8,279,216 (31.58 MB)

 Trainable params: 8,279,216 (31.58 MB)

 Non-trainable params: 0 (0.00 B)

**El corpus entero, 2 épocas, como en los apuntes** (81 minutos, una sola vez; después se lee de disco). Sin validación: aquí no hay etiquetas, y lo que importa es cómo funcione después M3 con PhraseBank. Los datos se pasan con `tf.data` y `shuffle(1000)`, como en los apuntes; ese búfer resultó ser un error con nuestro corpus (sección 10).

In [34]:
RUTA_MLM = CARPETA_MODELOS / "m3_preentrenado.keras"
RUTA_HISTORIAL_MLM = CARPETA_MODELOS / "m3_preentrenado_historial.json"
EPOCAS_PREENTRENAMIENTO = 2

if RUTA_MLM.exists() and RUTA_HISTORIAL_MLM.exists():
    bert_masked_model = keras.models.load_model(RUTA_MLM, custom_objects=CAPAS_PROPIAS)
    historial_mlm = json.loads(RUTA_HISTORIAL_MLM.read_text(encoding="utf-8"))
    print("Ya preentrenado, se lee de disco")
else:
    mlm_ds = tf.data.Dataset.from_tensor_slices((x_masked_train, y_masked_labels, sample_weights))
    mlm_ds = mlm_ds.shuffle(1000, seed=SEMILLA).batch(BATCH_SIZE)
    bert_masked_model = create_masked_language_bert_model()
    inicio = time.time()
    historia = bert_masked_model.fit(mlm_ds, epochs=EPOCAS_PREENTRENAMIENTO)
    historial_mlm = {"loss": [float(valor) for valor in historia.history["loss"]],
                     "minutos": round((time.time() - inicio) / 60, 1)}
    bert_masked_model.save(RUTA_MLM)
    RUTA_HISTORIAL_MLM.write_text(json.dumps(historial_mlm), encoding="utf-8")
    print("Preentrenado y guardado")

print(f"Pérdida por época: {np.round(historial_mlm['loss'], 4).tolist()} · duración: {historial_mlm['minutos']} min")

Ya preentrenado, se lee de disco
Pérdida por época: [0.2059, 0.1658] · duración: 80.6 min


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/layer.py:431: UserWarning:

`build()` was called on layer 'positional_embedding_5', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/layer.py:431: UserWarning:

`build()` was called on layer 'encoder_0', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.



La pérdida de Keras promedia también las posiciones con peso 0, así que es poco intuitiva. Se mide mejor así: se tapan las frases de la validación de PhraseBank (que nunca ha visto) y se cuenta cuántas veces acierta la palabra, frente a decir siempre la palabra más tapada del corpus.

In [35]:
numeros_validacion_frases = vectorize_layer(validacion["frase"].tolist()).numpy()
np.random.seed(SEMILLA)
x_tapadas_val, y_tapadas_val, pesos_tapadas_val = get_masked_input_and_labels(numeros_validacion_frases)
posiciones = pesos_tapadas_val > 0

prediccion_palabras = bert_masked_model.predict(x_tapadas_val, batch_size=BATCH_SIZE, verbose=0).argmax(axis=-1)
acierto_mlm = (prediccion_palabras[posiciones] == y_tapadas_val[posiciones]).mean()
palabra_frecuente = np.bincount(y_masked_labels[sample_weights > 0]).argmax()   # la más tapada en el corpus
acierto_frecuente = (y_tapadas_val[posiciones] == palabra_frecuente).mean()
print(f"Palabras tapadas en la validación: {posiciones.sum():,}")
print(f"Acierto del modelo preentrenado: {acierto_mlm:.1%} · decir siempre '{vocabulario[palabra_frecuente]}': {acierto_frecuente:.1%}")

Palabras tapadas en la validación: 2,537
Acierto del modelo preentrenado: 20.7% · decir siempre 'of': 3.7%


Y unos ejemplos escritos a mano, con una palabra tapada: las cinco palabras que la red ve más probables en el hueco.

In [36]:
def completar(frase: str, cuantas: int = 5) -> list[str]:
    """Las palabras más probables para el primer [mask] de una frase, según el modelo preentrenado."""
    numeros = vectorize_layer([frase]).numpy()
    posicion = int(np.where(numeros[0] == mask_token_id)[0][0])
    probabilidades = bert_masked_model.predict(numeros, verbose=0)[0, posicion]
    return [vocabulario[n] for n in probabilidades.argsort()[::-1][:cuantas]]


for frase in ["The company reported a net [mask] of EUR 5 million .",
              "Shares of the bank [mask] 10 % after the results .",
              "Operating profit rose to EUR 13.1 mn from EUR 8.7 mn in the corresponding [mask] in 2007 .",
              "We may not be able to [mask] our debt obligations ."]:
    print(f"{frase}\n   -> {completar(frase)}")

The company reported a net [mask] of EUR 5 million .
   -> ['loss', 'gain', 'charge', 'income', 'sales']
Shares of the bank [mask] 10 % after the results .
   -> ['was', 'note', 'is', 'at', 'notes']
Operating profit rose to EUR 13.1 mn from EUR 8.7 mn in the corresponding [mask] in 2007 .
   -> ['period', 'periods', 'increase', 'change', 'quarter']
We may not be able to [mask] our debt obligations .
   -> ['execute', 'maintain', 'refinance', 'access', 'secure']


**Conclusión del preentrenamiento.** Se tapa el 14,1 % de las palabras del corpus (algo menos del 15 % porque *the* nunca se tapa). En 2 épocas la pérdida baja de 0,206 a 0,166, todavía bajando al terminar. En frases nunca vistas, la red acierta la palabra tapada el **20,7 %** de las veces, frente al 3,7 % de decir siempre *of*: ha aprendido algo del lenguaje financiero. Completa bien el vocabulario contable (*net loss*, *corresponding period*, *refinance our debt*), pero en *Shares of the bank [mask] 10 %* no propone *rose* ni *fell*, que es justo la clase de palabra que lleva el sentimiento.

## 8. M3, segunda fase: ¿ayuda el preentrenamiento a clasificar el sentimiento?

Como en `bert_simplificado.ipynb`, en dos fases y con las mismas 2.901 frases que M2:

1. **Encoder congelado** (`trainable = False`): solo aprende la parte final, la misma que la de M2.
2. **Todo descongelado:** se sigue desde la fase 1 con un `Adam()` nuevo, y ahora también se ajusta el encoder.

**M3 es el modelo de la fase 2**, como en los apuntes.

**Diferencias con los apuntes:** corte a la salida de `encoder_0` (el mismo punto de la red que `encoder_0/ffn_layernormalization`); 3 clases; `EarlyStopping` sobre la validación en las dos fases en lugar de 2 épocas fijas mirando el test; sin pesos de clase, como M2.

### 8.1 El encoder preentrenado, congelado

Se corta la red preentrenada a la salida del encoder (`pretrained_bert_model`, como en los apuntes) y se congela. Con el encoder congelado, el clasificador (`create_classifier_bert_model`) solo entrena los 8.451 parámetros de la parte final.

In [37]:
mlm_model = keras.models.load_model(RUTA_MLM, custom_objects=CAPAS_PROPIAS)
pretrained_bert_model = keras.Model(mlm_model.input, mlm_model.get_layer("encoder_0").output, name="bert_preentrenado")

# Freeze it
pretrained_bert_model.trainable = False

In [38]:
def create_classifier_bert_model():
    """Clasificador de bert_simplificado sobre el encoder preentrenado, con la misma parte final que M2."""
    keras.utils.set_random_seed(SEMILLA)   # mismos pesos iniciales de la parte final cada vez
    inputs = layers.Input((MAX_LEN,), dtype="int64")
    sequence_output = pretrained_bert_model(inputs)
    pooled_output = layers.GlobalMaxPooling1D()(sequence_output)
    hidden_layer = layers.Dense(64, activation="relu")(pooled_output)
    outputs = layers.Dense(3, activation="softmax")(hidden_layer)
    classifer_model = keras.Model(inputs, outputs, name="classification")
    optimizer = keras.optimizers.Adam()
    classifer_model.compile(optimizer=optimizer, loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return classifer_model


create_classifier_bert_model().summary()

Model: "classification"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_8 (InputLayer)      │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bert_preentrenado (Functional)  │ (None, 64, 128)        │     4,409,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_1          │ (None, 128)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,417,667 (16.85 MB)

 Trainable params: 8,451 (33.01 KB)

 Non-trainable params: 4,409,216 (16.82 MB)

In [39]:
m3_congelado, historial_congelado = entrenar(create_classifier_bert_model, "m3_congelado")
dibujar_perdidas(historial_congelado, "M3, fase 1 (encoder congelado): pérdida por época")

m3_congelado: ya entrenado, se lee de disco (12 épocas, la mejor la 9)


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/saving/saving_lib.py:869: UserWarning:

Skipping variable loading for optimizer 'adam_4', because it has 46 variables whereas the saved optimizer has 10 variables. 



### 8.2 Todo descongelado

Se parte del modelo de la fase 1 guardado en disco (con su mejor época), se descongela el encoder, se compila con un `Adam()` nuevo y se sigue entrenando.

In [40]:
def descongelar(ruta_fase_1=CARPETA_MODELOS / "m3_congelado.keras"):
    """El clasificador de la fase 1 (leído de disco) con el encoder descongelado y un Adam nuevo, como en los apuntes."""
    keras.utils.set_random_seed(SEMILLA)
    classifer_model = keras.models.load_model(ruta_fase_1, custom_objects=CAPAS_PROPIAS, compile=False)
    # Unfreeze the BERT model for fine-tuning
    classifer_model.get_layer("bert_preentrenado").trainable = True
    optimizer = keras.optimizers.Adam()
    classifer_model.compile(optimizer=optimizer, loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return classifer_model


m3, historial_m3 = entrenar(descongelar, "m3_descongelado")
dibujar_perdidas(historial_m3, "M3, fase 2 (encoder descongelado): pérdida por época")

m3_descongelado: ya entrenado, se lee de disco (4 épocas, la mejor la 1)


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/layer.py:431: UserWarning:

`build()` was called on layer 'positional_embedding_5', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/layer.py:431: UserWarning:

`build()` was called on layer 'encoder_0', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.



### 8.3 M3 frente a M2 en la validación

El test sigue sin mirarse: se usa una sola vez, en el paso 9.

In [41]:
evaluacion_m3_congelado = evaluar_clasificacion(reales_validacion, predecir(m3_congelado, x_validacion))
evaluacion_m3 = evaluar_clasificacion(reales_validacion, predecir(m3, x_validacion))

pd.DataFrame({
    nombre: {"F1 macro": e["f1_macro"], "IC 95 % inferior": e["f1_ic_inferior"], "IC 95 % superior": e["f1_ic_superior"],
             "exactitud": e["exactitud"]}
    for nombre, e in [("Siempre neutral", evaluacion_neutral), ("M1 diccionario", evaluacion_m1),
                      ("M2 Transformer", evaluacion_m2), ("M3 fase 1 (congelado)", evaluacion_m3_congelado),
                      ("M3 BERT (fase 2)", evaluacion_m3)]
}).T.round(3)

,F1 macro,IC 95 % inferior,IC 95 % superior,exactitud
Siempre neutral,0.248,0.240,0.257,0.594
M1 diccionario,0.509,0.472,0.545,0.627
M2 Transformer,0.675,0.639,0.712,0.754
M3 fase 1 (congelado),0.516,0.475,0.557,0.689
M3 BERT (fase 2),0.612,0.570,0.651,0.727


In [42]:
print(f"M3 (validación, {evaluacion_m3['frases']} frases)")
print(f"F1 macro: {evaluacion_m3['f1_macro']:.3f}  (IC 95 %: {evaluacion_m3['f1_ic_inferior']:.3f} – {evaluacion_m3['f1_ic_superior']:.3f})")
print(f"Exactitud: {evaluacion_m3['exactitud']:.3f}")
evaluacion_m3["matriz_confusion"]

M3 (validación, 967 frases)
F1 macro: 0.612  (IC 95 %: 0.570 – 0.651)
Exactitud: 0.727


predicha,negative,neutral,positive
real,,,
negative,36,53,32
neutral,3,530,41
positive,2,133,137


**Conclusión: en la validación, preentrenar no ha ayudado.** M3 llega a un **F1 macro de 0,612** (IC 95 %: 0,57 – 0,65), frente a 0,675 de M2 (0,64 – 0,71): no mejora a M2, y probablemente es algo peor.

- **Fase 1 (congelado):** 12 épocas, la mejor la 9, y un F1 de 0,516, casi el de M1. Lo que el encoder aprendió adivinando palabras separa poco el sentimiento por sí solo (como se vio en el paso 7). Los apuntes ven lo mismo: con el modelo congelado el resultado es pobre.
- **Fase 2 (descongelado):** la mejor época es la primera (pérdida de validación 0,655), y después se sobreajusta como M2.
- **Menor pérdida, pero menor F1:** la pérdida de M3 (0,655) es algo menor que la de M2 (0,663), pero M3 manda más frases a "neutral" (133 de 272 positivas y 53 de 121 negativas, frente a 99 y 42 en M2). `EarlyStopping` elige por pérdida, criterio fijado antes de ver los resultados.
- **Por qué puede pasar:** este preentrenamiento es diminuto frente al de BERT (una capa, 17 millones de palabras y 2 épocas, frente a 110 millones de parámetros y 3.300 millones de palabras). Es lo que dicen los propios apuntes: un modelo preentrenado pobre aporta poco.

## 9. ¿Qué modelo mide mejor el sentimiento? El examen final con el test

Con M1, M2 y M3 ya fijados con la validación, se miden **una sola vez** con las 968 frases del test. Dos preguntas:

1. **¿Aprender de ejemplos supera a una lista de palabras hecha por expertos?** M2 − M1.
2. **¿Aporta preentrenar?** M3 − M2.

Se mide la **diferencia de F1 macro con su propio intervalo**, remuestreando las mismas frases para los dos modelos (*bootstrap* emparejado, `comparar_f1`). Si el intervalo contiene el 0, la diferencia no se presenta como mejora. **Sensibilidad:** se repite con las 448 frases del test de AllAgree, en las que todos los anotadores coincidieron.

**Diferencia con los apuntes:** allí el test se usa durante el entrenamiento; aquí se mira solo al final, una vez.

### 9.1 Lo que dice cada modelo

Para cada frase de la validación y del test, la etiqueta real y la de cada modelo, guardadas en `data/cache/predicciones_phrasebank.parquet`.

In [43]:
MODELOS = ["siempre neutral", "M1 diccionario", "M2 Transformer", "M3 BERT"]


def predicciones(parte: str) -> pd.DataFrame:
    """Las frases de una parte de la partición, con su etiqueta real y la que da cada modelo."""
    tabla = particion[particion["parte"] == parte].copy()
    x, _ = preparar(parte)
    tabla["siempre neutral"] = "neutral"
    tabla["M1 diccionario"] = [clasificar_tono(tono_loughran_mcdonald(frase, palabras_negativas, palabras_positivas), umbral_m1)
                               for frase in tabla["frase"]]
    tabla["M2 Transformer"] = predecir(m2, x)
    tabla["M3 BERT"] = predecir(m3, x)
    return tabla


predicciones_phrasebank = pd.concat([predicciones("validacion"), predicciones("test")], ignore_index=True)
predicciones_phrasebank.to_parquet(CARPETA_CACHE / "predicciones_phrasebank.parquet", index=False)
test = predicciones_phrasebank[predicciones_phrasebank["parte"] == "test"]
print(f"Test: {len(test)} frases, {test['en_allagree'].sum()} de ellas en AllAgree")

Test: 968 frases, 448 de ellas en AllAgree


### 9.2 Resultados en el test

In [44]:
def tabla_resultados(frases: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """F1 macro con su intervalo y exactitud de cada modelo sobre unas frases; también las evaluaciones completas."""
    evaluaciones = {modelo: evaluar_clasificacion(frases["etiqueta"], frases[modelo]) for modelo in MODELOS}
    tabla = pd.DataFrame({
        modelo: {"F1 macro": e["f1_macro"], "IC 95 % inferior": e["f1_ic_inferior"], "IC 95 % superior": e["f1_ic_superior"],
                 "exactitud": e["exactitud"]}
        for modelo, e in evaluaciones.items()
    }).T.round(3)
    return tabla, evaluaciones


tabla_test, evaluaciones_test = tabla_resultados(test)
tabla_test

,F1 macro,IC 95 % inferior,IC 95 % superior,exactitud
siempre neutral,0.248,0.240,0.256,0.594
M1 diccionario,0.502,0.463,0.539,0.631
M2 Transformer,0.673,0.632,0.709,0.746
M3 BERT,0.607,0.563,0.646,0.728


Las dos preguntas, con la diferencia de F1 macro y su intervalo:

In [45]:
PREGUNTAS = {
    "¿Aprender de ejemplos supera al diccionario? (M2 − M1)": ("M2 Transformer", "M1 diccionario"),
    "¿Aporta preentrenar? (M3 − M2)": ("M3 BERT", "M2 Transformer"),
}


def tabla_diferencias(frases: pd.DataFrame) -> pd.DataFrame:
    """Diferencia de F1 macro de cada pregunta, con su intervalo y si el 0 queda fuera de él."""
    filas = {}
    for pregunta, (modelo_a, modelo_b) in PREGUNTAS.items():
        d = comparar_f1(frases["etiqueta"], frases[modelo_a], frases[modelo_b])
        filas[pregunta] = {"diferencia": round(d["diferencia"], 3), "IC 95 % inferior": round(d["ic_inferior"], 3),
                           "IC 95 % superior": round(d["ic_superior"], 3),
                           "¿concluyente?": "sí" if d["ic_inferior"] > 0 or d["ic_superior"] < 0 else "no (el 0 está dentro)"}
    return pd.DataFrame(filas).T


tabla_diferencias(test)

,diferencia,IC 95 % inferior,IC 95 % superior,¿concluyente?
¿Aprender de ejemplos supera al diccionario? (M2 − M1),0.171,0.112,0.228,sí
¿Aporta preentrenar? (M3 − M2),-0.066,-0.112,-0.021,sí


Las matrices de confusión de los tres modelos, una al lado de otra (filas = etiqueta real, columnas = la del modelo):

In [46]:
pd.concat({modelo: evaluaciones_test[modelo]["matriz_confusion"] for modelo in MODELOS[1:]}, axis=1)

M1 diccionario                  M2 Transformer                   \
predicha       negative neutral positive       negative neutral positive   
real                                                                       
negative             47      71        3             57      31       33   
neutral              37     501       37             18     502       55   
positive             19     190       63              5     104      163   

          M3 BERT                   
predicha negative neutral positive  
real                                
negative       37      51       33  
neutral         5     538       32  
positive        6     136      130

### 9.3 Sensibilidad: solo las frases en las que todos los anotadores coinciden (AllAgree)

In [47]:
test_allagree = test[test["en_allagree"]]
tabla_allagree, _ = tabla_resultados(test_allagree)
display(tabla_allagree)
tabla_diferencias(test_allagree)

,F1 macro,IC 95 % inferior,IC 95 % superior,exactitud
siempre neutral,0.250,0.239,0.262,0.600
M1 diccionario,0.469,0.415,0.520,0.638
M2 Transformer,0.807,0.758,0.850,0.864
M3 BERT,0.717,0.660,0.765,0.810


,diferencia,IC 95 % inferior,IC 95 % superior,¿concluyente?
¿Aprender de ejemplos supera al diccionario? (M2 − M1),0.338,0.253,0.419,sí
¿Aporta preentrenar? (M3 − M2),-0.091,-0.151,-0.028,sí


**Conclusión del examen: aprender de ejemplos supera claramente al diccionario, y preentrenar, tal como se ha hecho aquí, no ayuda: empeora.**

| Modelo | F1 macro en el test (IC 95 %) | Exactitud |
|---|---|---|
| Siempre neutral | 0,248 (0,24 – 0,26) | 0,594 |
| M1 diccionario | 0,502 (0,46 – 0,54) | 0,631 |
| **M2 Transformer** | **0,673** (0,63 – 0,71) | 0,746 |
| M3 BERT | 0,607 (0,56 – 0,65) | 0,728 |

- **El test confirma la validación:** cada modelo queda a menos de 0,01 de su F1 en la validación.
- **M2 − M1 = +0,171** (IC 95 %: 0,11 – 0,23). El diccionario falla sobre todo por quedarse en "neutral" (190 de 272 positivas).
- **M3 − M2 = −0,066** (−0,11 a −0,02): el 0 queda fuera. La conclusión vale para este preentrenamiento, no para BERT en general.
- **Con AllAgree** el orden se mantiene y las distancias crecen: M2 0,807, M3 0,717 y M1 0,469.

**Nota, añadida después del examen:** tras mirar el test se encontró un error en el preentrenamiento de M3 (sección 10). El M3 oficial pasa a ser el corregido, como se decidió antes de ver su resultado: **0,586** en el test (0,54 – 0,62), con M3 − M2 = −0,087 (−0,13 a −0,05). La conclusión no cambia, y la tabla de arriba se deja tal como salió. Después se añadió M4 (sección 11).

## 10. Corrección: el preentrenamiento de M3 con el corpus bien mezclado

**Qué falló.** El corpus del notebook 06 guarda primero todos los titulares y después todas las frases de los informes. Con `shuffle(1000)`, como en los apuntes, cada frase solo se mueve unas miles de posiciones en un corpus de 695.136: en cada época, la red vio **primero solo titulares y después solo informes**. Se encontró el 16 nov, **después de mirar el test**.

**Qué se hace, decidido antes de ver el resultado:** repetir el preentrenamiento con una **permutación completa** del corpus (`shuffle` del tamaño del corpus, de nuevo en cada época) y el mismo ajuste del paso 8, sin cambiar nada más. **El M3 oficial pasa a ser el corregido, salga mejor o peor**, y su test se informa tal cual.

**Diferencia con los apuntes:** búfer de `shuffle` del tamaño del corpus en lugar de 1.000.

### 10.1 El fallo, con datos

El orden en que `tf.data` entrega las frases solo depende del búfer y de la semilla, así que se reproduce con sus posiciones. Si la mezcla es buena, cada décimo de la época tiene un 42 % de titulares, como el corpus:

In [48]:
fuente_corpus = pd.read_parquet(CARPETA_CACHE / "corpus_preentrenamiento.parquet", columns=["fuente"])["fuente"]
es_titular = (fuente_corpus == "titular").to_numpy()
print(f"Titulares: {es_titular.sum():,} ({es_titular.mean():.1%}), en las posiciones 0 a {np.where(es_titular)[0].max():,}"
      f" · frases de informes: {(~es_titular).sum():,}, detrás")


def titulares_por_decimo(buffer: int) -> list[int]:
    """% de titulares en los lotes de cada décimo de una época, en el orden de tf.data con ese búfer y SEMILLA."""
    posiciones_ds = tf.data.Dataset.range(len(es_titular)).shuffle(buffer, seed=SEMILLA).batch(BATCH_SIZE)
    orden = np.concatenate([lote.numpy() for lote in posiciones_ds])
    proporcion_por_lote = np.array([lote.mean() for lote in np.array_split(es_titular[orden], int(np.ceil(len(orden) / BATCH_SIZE)))])
    return [round(100 * decimo.mean()) for decimo in np.array_split(proporcion_por_lote, 10)]


pd.DataFrame({"shuffle(1000), sección 7": titulares_por_decimo(1000),
              "permutación completa, aquí": titulares_por_decimo(len(es_titular))},
             index=pd.Index(range(1, 11), name="% de titulares en el décimo")).T

Titulares: 294,536 (42.4%), en las posiciones 0 a 294,535 · frases de informes: 400,600, detrás


% de titulares en el décimo,1,2,3,4,5,6,7,8,9,10
"shuffle(1000), sección 7",100,100,100,100,24,0,0,0,0,0
"permutación completa, aquí",43,42,42,43,42,42,42,42,42,43


### 10.2 Preentrenamiento y ajuste corregidos

El mismo código de las secciones 7 y 8, con las mismas palabras tapadas, cambiando solo el búfer. Se guarda aparte (`m3_preentrenado_mezclado.keras`; unos 80 minutos, una sola vez). Se compara con el original en las palabras tapadas y en los mismos ejemplos, y después se ajusta en las mismas dos fases.

In [49]:
RUTA_MLM_MEZCLADO = CARPETA_MODELOS / "m3_preentrenado_mezclado.keras"
RUTA_HISTORIAL_MLM_MEZCLADO = CARPETA_MODELOS / "m3_preentrenado_mezclado_historial.json"

if RUTA_MLM_MEZCLADO.exists() and RUTA_HISTORIAL_MLM_MEZCLADO.exists():
    mlm_mezclado = keras.models.load_model(RUTA_MLM_MEZCLADO, custom_objects=CAPAS_PROPIAS)
    historial_mlm_mezclado = json.loads(RUTA_HISTORIAL_MLM_MEZCLADO.read_text(encoding="utf-8"))
    print("Ya preentrenado, se lee de disco")
else:
    mlm_ds = tf.data.Dataset.from_tensor_slices((x_masked_train, y_masked_labels, sample_weights))
    mlm_ds = mlm_ds.shuffle(len(x_masked_train), seed=SEMILLA, reshuffle_each_iteration=True).batch(BATCH_SIZE)
    mlm_mezclado = create_masked_language_bert_model()
    inicio = time.time()
    historia = mlm_mezclado.fit(mlm_ds, epochs=EPOCAS_PREENTRENAMIENTO)
    historial_mlm_mezclado = {"loss": [float(valor) for valor in historia.history["loss"]],
                              "minutos": round((time.time() - inicio) / 60, 1)}
    mlm_mezclado.save(RUTA_MLM_MEZCLADO)
    RUTA_HISTORIAL_MLM_MEZCLADO.write_text(json.dumps(historial_mlm_mezclado), encoding="utf-8")
    print("Preentrenado y guardado")

print(f"Pérdida por época · corregido: {np.round(historial_mlm_mezclado['loss'], 4).tolist()} ({historial_mlm_mezclado['minutos']} min)"
      f" · original: {np.round(historial_mlm['loss'], 4).tolist()}")

Ya preentrenado, se lee de disco
Pérdida por época · corregido: [0.2256, 0.1662] (76.6 min) · original: [0.2059, 0.1658]


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/layer.py:431: UserWarning:

`build()` was called on layer 'positional_embedding_3', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/layer.py:431: UserWarning:

`build()` was called on layer 'encoder_0', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.



In [50]:
acierto_mlm_mezclado = (mlm_mezclado.predict(x_tapadas_val, batch_size=BATCH_SIZE, verbose=0).argmax(axis=-1)[posiciones]
                        == y_tapadas_val[posiciones]).mean()
print(f"Acierto en las palabras tapadas de la validación · corregido: {acierto_mlm_mezclado:.1%} · original: {acierto_mlm:.1%}"
      f" · decir siempre '{vocabulario[palabra_frecuente]}': {acierto_frecuente:.1%}\n")

for frase in ["The company reported a net [mask] of EUR 5 million .",
              "Shares of the bank [mask] 10 % after the results .",
              "Operating profit rose to EUR 13.1 mn from EUR 8.7 mn in the corresponding [mask] in 2007 .",
              "We may not be able to [mask] our debt obligations ."]:
    numeros = vectorize_layer([frase]).numpy()
    posicion = int(np.where(numeros[0] == mask_token_id)[0][0])
    probabilidades = mlm_mezclado.predict(numeros, verbose=0)[0, posicion]
    print(f"{frase}\n   corregido -> {[vocabulario[n] for n in probabilidades.argsort()[::-1][:5]]}\n   original  -> {completar(frase)}")

Acierto en las palabras tapadas de la validación · corregido: 20.9% · original: 20.7% · decir siempre 'of': 3.7%

The company reported a net [mask] of EUR 5 million .
   corregido -> ['dividend', 'sales', 'loss', 'charge', 'investment']
   original  -> ['loss', 'gain', 'charge', 'income', 'sales']
Shares of the bank [mask] 10 % after the results .
   corregido -> ['up', 'of', 'down', 'market', 'is']
   original  -> ['was', 'note', 'is', 'at', 'notes']
Operating profit rose to EUR 13.1 mn from EUR 8.7 mn in the corresponding [mask] in 2007 .
   corregido -> ['period', 'periods', 'year', 'increase', 'facility']
   original  -> ['period', 'periods', 'increase', 'change', 'quarter']
We may not be able to [mask] our debt obligations .
   corregido -> ['meet', 'maintain', 'refinance', 'satisfy', 'pay']
   original  -> ['execute', 'maintain', 'refinance', 'access', 'secure']


In [51]:
pretrained_bert_model = keras.Model(mlm_mezclado.input, mlm_mezclado.get_layer("encoder_0").output, name="bert_preentrenado")
pretrained_bert_model.trainable = False

m3_corregido_congelado, historial_corregido_congelado = entrenar(create_classifier_bert_model, "m3_mezclado_congelado")
dibujar_perdidas(historial_corregido_congelado, "M3 corregido, fase 1 (encoder congelado): pérdida por época")

m3_mezclado_congelado: ya entrenado, se lee de disco (15 épocas, la mejor la 12)


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/saving/saving_lib.py:869: UserWarning:

Skipping variable loading for optimizer 'adam_5', because it has 46 variables whereas the saved optimizer has 10 variables. 



In [52]:
m3_corregido, historial_m3_corregido = entrenar(lambda: descongelar(CARPETA_MODELOS / "m3_mezclado_congelado.keras"),
                                                "m3_mezclado_descongelado")
dibujar_perdidas(historial_m3_corregido, "M3 corregido, fase 2 (encoder descongelado): pérdida por época")

m3_mezclado_descongelado: ya entrenado, se lee de disco (4 épocas, la mejor la 1)


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/layer.py:431: UserWarning:

`build()` was called on layer 'positional_embedding_3', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/layer.py:431: UserWarning:

`build()` was called on layer 'encoder_0', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.



### 10.3 El resultado, en la validación y en el test

En la validación, junto a M2 y al M3 original. En el test, el M3 corregido se añade a las predicciones guardadas y se repite solo su comparación con M2, con todas las frases y con las de AllAgree.

In [53]:
evaluacion_m3_corregido = evaluar_clasificacion(reales_validacion, predecir(m3_corregido, x_validacion))
evaluacion_m3_corregido_congelado = evaluar_clasificacion(reales_validacion, predecir(m3_corregido_congelado, x_validacion))

pd.DataFrame({
    nombre: {"F1 macro": e["f1_macro"], "IC 95 % inferior": e["f1_ic_inferior"], "IC 95 % superior": e["f1_ic_superior"],
             "exactitud": e["exactitud"]}
    for nombre, e in [("M2 Transformer", evaluacion_m2), ("M3 original (fase 2)", evaluacion_m3),
                      ("M3 corregido, fase 1 (congelado)", evaluacion_m3_corregido_congelado),
                      ("M3 corregido (fase 2)", evaluacion_m3_corregido)]
}).T.round(3)

,F1 macro,IC 95 % inferior,IC 95 % superior,exactitud
M2 Transformer,0.675,0.639,0.712,0.754
M3 original (fase 2),0.612,0.570,0.651,0.727
"M3 corregido, fase 1 (congelado)",0.475,0.440,0.512,0.656
M3 corregido (fase 2),0.600,0.561,0.640,0.714


In [54]:
predicciones_phrasebank["M3 BERT (corregido)"] = (predecir(m3_corregido, preparar("validacion")[0])
                                                  + predecir(m3_corregido, preparar("test")[0]))
predicciones_phrasebank.to_parquet(CARPETA_CACHE / "predicciones_phrasebank.parquet", index=False)
test = predicciones_phrasebank[predicciones_phrasebank["parte"] == "test"]

resultados_m3 = {}
for nombre_examen, frases_examen in [("test", test), ("test, solo AllAgree", test[test["en_allagree"]])]:
    for modelo in ["M2 Transformer", "M3 BERT", "M3 BERT (corregido)"]:
        e = evaluar_clasificacion(frases_examen["etiqueta"], frases_examen[modelo])
        resultados_m3[(nombre_examen, modelo)] = {"F1 macro": e["f1_macro"], "IC 95 % inferior": e["f1_ic_inferior"],
                                                   "IC 95 % superior": e["f1_ic_superior"], "exactitud": e["exactitud"]}
pd.DataFrame(resultados_m3).T.round(3)

F1 macro  IC 95 % inferior  \
test                M2 Transformer          0.673             0.632   
                    M3 BERT                 0.607             0.563   
                    M3 BERT (corregido)     0.586             0.544   
test, solo AllAgree M2 Transformer          0.807             0.758   
                    M3 BERT                 0.717             0.660   
                    M3 BERT (corregido)     0.739             0.686   

                                         IC 95 % superior  exactitud  
test                M2 Transformer                  0.709      0.746  
                    M3 BERT                         0.646      0.728  
                    M3 BERT (corregido)             0.623      0.696  
test, solo AllAgree M2 Transformer                  0.850      0.864  
                    M3 BERT                         0.765      0.810  
                    M3 BERT (corregido)             0.785      0.817

In [55]:
diferencias_m3_corregido = {}
for nombre_examen, frases_examen in [("test", test), ("test, solo AllAgree", test[test["en_allagree"]])]:
    d = comparar_f1(frases_examen["etiqueta"], frases_examen["M3 BERT (corregido)"], frases_examen["M2 Transformer"])
    diferencias_m3_corregido[f"¿Aporta preentrenar? (M3 corregido − M2), {nombre_examen}"] = {
        "diferencia": round(d["diferencia"], 3), "IC 95 % inferior": round(d["ic_inferior"], 3),
        "IC 95 % superior": round(d["ic_superior"], 3),
        "¿concluyente?": "sí" if d["ic_inferior"] > 0 or d["ic_superior"] < 0 else "no (el 0 está dentro)"}
pd.DataFrame(diferencias_m3_corregido).T

,diferencia,IC 95 % inferior,IC 95 % superior,¿concluyente?
"¿Aporta preentrenar? (M3 corregido − M2), test",-0.087,-0.131,-0.047,sí
"¿Aporta preentrenar? (M3 corregido − M2), test, solo AllAgree",-0.068,-0.124,-0.018,sí


**Conclusión de la corrección: el fallo era real, pero no explicaba el mal resultado de M3.** Con el corpus bien mezclado, M3 queda igual o algo peor que antes, y sigue claramente por debajo de M2. El M3 oficial es el corregido, como se decidió antes de verlo.

| | F1 macro en la validación | F1 macro en el test (IC 95 %) | M3 − M2 en el test (IC 95 %) |
|---|---|---|---|
| M2 Transformer | 0,675 | 0,673 (0,63 – 0,71) | — |
| M3 original | 0,612 | 0,607 (0,56 – 0,65) | −0,066 (−0,11 a −0,02) |
| **M3 corregido (oficial)** | 0,600 | **0,586** (0,54 – 0,62) | **−0,087** (−0,13 a −0,05) |

- **El fallo se confirmó con datos (10.1):** con `shuffle(1000)`, los cuatro primeros décimos de cada época eran solo titulares y los cinco últimos solo frases de informes. Con la permutación completa, todos los décimos tienen un 42 % de titulares, la proporción del corpus.
- **El preentrenamiento sale prácticamente igual (10.2):** la pérdida acaba en 0,166 en los dos y el acierto en palabras tapadas es del 20,9 %, frente al 20,7 %. Cambian algunos ejemplos: ahora propone *up* y *down* en *Shares of the bank [mask] 10 %*, aunque detrás de *of*.
- **El ajuste se comporta igual (10.2):** la fase 1 se queda en 0,475 (0,516 antes) y la fase 2 vuelve a tener la mejor época en la primera y a sobreajustar después.
- **¿Aporta preentrenar? Sigue siendo que no.** En el test, M3 corregido − M2 = −0,087, y el 0 queda fuera. Con AllAgree, −0,068 (−0,12 a −0,02). Respecto al M3 original, la diferencia (−0,02) está dentro de lo que cambia entre dos entrenamientos. Su error principal ahora son las negativas: acierta 34 de 121 y manda 53 a neutral y 34 a positiva.

Con esto queda cerrada la comparación de M1, M2 y M3. El notebook 08 usa el M3 corregido (`m3_mezclado_descongelado.keras`).

## 11. M4: ¿y si se parte de un BERT financiero preentrenado por otros?

Lo habitual en la práctica no es preentrenar un BERT propio, como M3, sino partir de uno preentrenado a gran escala y ajustarlo con las frases etiquetadas. Lo sugirió el tutor después de ver M3.

**Transparencia.** M4 se añade **después de haber mirado el test de M1, M2 y M3**, por sugerencia del tutor y no por los resultados. Todas sus decisiones (longitud, velocidad de aprendizaje, cabeza y una sola fase) se fijaron antes de entrenarlo, y su test se mira una sola vez. Nada de M1, M2 ni M3 cambia.

**FinBERT** (Yang, Uy y Huang, 2020), en su versión **solo preentrenada** (`yiyanghkust/finbert-pretrain`, licencia Apache-2.0): un BERT base de 110 M de parámetros preentrenado con 4.900 M de palabras de informes 10-K y 10-Q, conferencias de resultados e informes de analistas. **No ha visto PhraseBank.** Se excluyen `ProsusAI/finbert` (ajustado con PhraseBank: habría visto el test) y `finbert-tone` (ya es un clasificador de sentimiento). Plan B, decidido antes y no usado: `nlpaueb/sec-bert-base`.

**Qué se puede concluir.** M3 − M2 sigue siendo la única comparación limpia de "¿aporta preentrenar?". **M4 − M2 y M4 − M3 comparan recetas completas**: entre M3 y M4 cambian a la vez la red (110 M de parámetros frente a 8 M), el corpus, el tokenizador, la cabeza y el ajuste.

**Diferencias con los apuntes**, todas por partir de un modelo que no está en ellos:
- El modelo se carga con **KerasHub** (`BertBackbone`) en lugar de construirse con las capas de clase.
- **El tokenizador es el de FinBERT** (WordPiece): los pesos solo tienen sentido con su vocabulario.
- **Adam con 2e-5** en lugar de 0,001: pasos tan grandes destrozarían lo preentrenado. Es la de FinBERT y del BERT original.
- **Una sola fase con toda la red**, lo habitual con un BERT preentrenado. En M3 la fase congelada no aportó.
- **Cabeza: salida de `[CLS]` + Dropout(0,1) + Dense(3)**, la estándar de BERT. El máximo de todas las posiciones de los apuntes incluye el relleno y cambia según la longitud a la que se rellene; `[CLS]` no.

### 11.1 Descarga y conversión

Se descarga el único commit del repositorio (`88ab954`) y se comprueba la huella SHA-256 de los pesos. KerasHub no lo lee tal cual (al `config.json` de 2020 le falta `model_type`, y los pesos están en el formato de PyTorch), así que se guarda una copia en `data/modelos/finbert_pretrain/` con `model_type` añadido y los pesos del encoder en `safetensors`. `torch` se usa solo para esa conversión, una vez; después se lee de disco.

In [56]:
REPO_FINBERT = "yiyanghkust/finbert-pretrain"
COMMIT_FINBERT = "88ab954a39ea6d3ce2b62cff086dd5ad1172c664"   # el único commit del repositorio (octubre de 2022)
SHA256_PESOS_FINBERT = "46dd5b5cbf7141b0c5d882516243abf71cfa4a27e57023c43290d655fccfb48c"
CARPETA_FINBERT = CARPETA_MODELOS / "finbert_pretrain"

if (CARPETA_FINBERT / "model.safetensors").exists():
    print("FinBERT ya descargado y convertido, se lee de disco")
else:
    import torch                                   # solo aquí: para leer una vez los pesos en el formato de PyTorch
    from safetensors.torch import save_file

    descargados = {nombre: Path(hf_hub_download(REPO_FINBERT, nombre, revision=COMMIT_FINBERT))
                   for nombre in ["config.json", "vocab.txt", "pytorch_model.bin"]}
    huella = hashlib.sha256(descargados["pytorch_model.bin"].read_bytes()).hexdigest()
    if huella != SHA256_PESOS_FINBERT:
        raise ValueError(f"Los pesos descargados no son los esperados: SHA-256 {huella}")

    pesos = torch.load(descargados["pytorch_model.bin"], map_location="cpu", weights_only=True)
    pesos_encoder = {nombre: tensor.contiguous() for nombre, tensor in pesos.items() if nombre.startswith("bert.")}
    print(f"Tensores en el fichero: {len(pesos)} · del encoder: {len(pesos_encoder)}"
          f" · no se copian: {[nombre for nombre in pesos if not nombre.startswith('bert.')]}")

    CARPETA_FINBERT.mkdir(exist_ok=True)
    configuracion = json.loads(descargados["config.json"].read_text(encoding="utf-8"))
    configuracion["model_type"] = "bert"           # lo que KerasHub necesita para reconocer el formato
    (CARPETA_FINBERT / "config.json").write_text(json.dumps(configuracion, indent=1), encoding="utf-8")
    shutil.copy(descargados["vocab.txt"], CARPETA_FINBERT / "vocab.txt")
    save_file(pesos_encoder, str(CARPETA_FINBERT / "model.safetensors"))   # el último: si algo falla antes, no queda a medias
    print("FinBERT descargado y convertido")

print("Ficheros:", {f.name: f"{f.stat().st_size / 1e6:.1f} MB" for f in sorted(CARPETA_FINBERT.iterdir())})
print((CARPETA_FINBERT / "config.json").read_text(encoding="utf-8"))

FinBERT ya descargado y convertido, se lee de disco
Ficheros: {'config.json': '0.0 MB', 'model.safetensors': '439.0 MB', 'vocab.txt': '0.2 MB'}
{
 "architectures": [
  "BertForMaskedLM"
 ],
 "attention_probs_dropout_prob": 0.1,
 "hidden_act": "gelu",
 "hidden_dropout_prob": 0.1,
 "hidden_size": 768,
 "initializer_range": 0.02,
 "intermediate_size": 3072,
 "max_position_embeddings": 512,
 "num_attention_heads": 12,
 "num_hidden_layers": 12,
 "type_vocab_size": 2,
 "vocab_size": 30873,
 "model_type": "bert"
}


### 11.2 El tokenizador y la longitud

FinVocab en minúsculas, 30.873 piezas: las palabras frecuentes son una pieza y las raras se parten en trozos (`##…`). Cada frase lleva `[CLS]` al principio y `[SEP]` al final; con **136 tokens** no se corta ninguna frase (la más larga ocupa 134).

In [57]:
SECUENCIA_FINBERT = 136   # tokens por frase, con [CLS] y [SEP]; la frase más larga de PhraseBank ocupa 134

tokenizador_finbert = keras_hub.tokenizers.BertTokenizer(vocabulary=str(CARPETA_FINBERT / "vocab.txt"), lowercase=True)
preprocesador_finbert = keras_hub.models.BertTextClassifierPreprocessor(tokenizador_finbert, sequence_length=SECUENCIA_FINBERT)

for ejemplo in ["Operating profit rose to EUR 13.1 mn from EUR 8.7 mn in the corresponding period in 2007 .",
                "We may not be able to refinance our debt obligations ."]:
    print([tokenizador_finbert.id_to_token(int(numero)) for numero in tokenizador_finbert([ejemplo])[0]])

tokens_por_frase = pd.Series([len(tokens) + 2 for tokens in tokenizador_finbert(particion["frase"].tolist())],
                             index=particion.index)   # + 2: [CLS] y [SEP]
print(f"\nTokens por frase en PhraseBank: media {tokens_por_frase.mean():.0f}, máximo {tokens_por_frase.max()}"
      f" · más de 64: {(tokens_por_frase > 64).sum()} · más de 128: {(tokens_por_frase > 128).sum()}"
      f" · vocabulario: {tokenizador_finbert.vocabulary_size():,} piezas")
print("La más larga:", particion.loc[tokens_por_frase.idxmax(), "frase"])

['operating', 'profit', 'rose', 'to', 'eur', '13', '.', '1', 'mn', 'from', 'eur', '8', '.', '7', 'mn', 'in', 'the', 'corresponding', 'period', 'in', '2007', '.']
['we', 'may', 'not', 'be', 'able', 'to', 'refinance', 'our', 'debt', 'obligations', '.']



Tokens por frase en PhraseBank: media 30, máximo 134 · más de 64: 75 · más de 128: 1 · vocabulario: 30,873 piezas
La más larga: Supported Nokia phones include : N96 , N95-8GB , N95 , N93-N931 , N92 , N85 , N82 , N81 , N80 , N79 , N78 , N77 , N76 , N75 , N73 , N72 , N71 , E90 , E71 , E70 , E66 , E65 , E62 , E61-E61i , E60 , E51 , E50 , Touch Xpress 5800 , 6220 Classic , 6210 Navigator , 6120 Classic , 6110 Navigator , 5700 , 5500 , 5320XM .


Las entradas de FinBERT son tres por frase (tokens, segmento y máscara de relleno), con la misma partición que los demás modelos:

In [58]:
def preparar_finbert(parte: str) -> tuple[dict, np.ndarray]:
    """Entradas de FinBERT de una parte de la partición (tokens, segmento y máscara de relleno) y sus etiquetas (0, 1, 2)."""
    tabla = particion[particion["parte"] == parte]
    entradas = preprocesador_finbert(tabla["frase"].tolist())
    y = tabla["etiqueta"].map({clase: numero for numero, clase in enumerate(CLASES_SENTIMIENTO)}).to_numpy()
    return {nombre: np.array(valores) for nombre, valores in entradas.items()}, y


x_entrenamiento_finbert, y_entrenamiento_finbert = preparar_finbert("entrenamiento")
x_validacion_finbert, y_validacion_finbert = preparar_finbert("validacion")
print("Entrenamiento:", {nombre: valores.shape for nombre, valores in x_entrenamiento_finbert.items()},
      "· validación:", len(y_validacion_finbert), "frases")

Entrenamiento: {'token_ids': (2901, 136), 'padding_mask': (2901, 136), 'segment_ids': (2901, 136)} · validación: 967 frases


### 11.3 El ajuste

Una sola fase, con lo mismo que el resto de redes: lotes de 32 barajados en cada época (KerasHub no los baraja por su cuenta), `EarlyStopping` con paciencia 3, sin pesos de clase y con la semilla. KerasHub construye BERT con una GELU aproximada; se cambia por la exacta del BERT original, con la que la salida coincide con la de la librería `transformers` (diferencia de 1e-5). Se guardan los pesos (`m4_finbert.weights.h5`, 1,3 GB con el estado del optimizador).

In [59]:
def crear_m4():
    """FinBERT preentrenado con la cabeza de clasificación estándar de BERT, compilado para ajustarlo entero."""
    keras.utils.set_random_seed(SEMILLA)
    backbone = keras_hub.models.BertBackbone.from_preset(str(CARPETA_FINBERT))
    # GELU exacta, la del BERT original con la que se preentrenó FinBERT (KerasHub usa una aproximación)
    for capa in backbone.transformer_layers:
        capa._feedforward_intermediate_dense.activation = keras.activations.gelu
    model = keras_hub.models.BertTextClassifier(backbone, num_classes=3, activation="softmax", dropout=0.1)
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=2e-5),   # Diferencia con los apuntes: 2e-5, no 0,001
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model


RUTA_M4 = CARPETA_MODELOS / "m4_finbert.weights.h5"
RUTA_HISTORIAL_M4 = CARPETA_MODELOS / "m4_finbert_historial.json"

m4 = crear_m4()
if RUTA_M4.exists() and RUTA_HISTORIAL_M4.exists():
    m4.load_weights(RUTA_M4)
    historial_m4 = json.loads(RUTA_HISTORIAL_M4.read_text(encoding="utf-8"))
    print(f"m4_finbert: ya ajustado, se lee de disco ({len(historial_m4['loss'])} épocas,"
          f" la mejor la {int(np.argmin(historial_m4['val_loss'])) + 1})")
else:
    inicio = time.time()
    parada = keras.callbacks.EarlyStopping(monitor="val_loss", patience=PACIENCIA, restore_best_weights=True)
    # KerasHub convierte los arrays en lotes sin barajarlos; se barajan aquí, como hace Keras con M2 y M3
    lotes_entrenamiento = (tf.data.Dataset.from_tensor_slices((x_entrenamiento_finbert, y_entrenamiento_finbert))
                           .shuffle(len(y_entrenamiento_finbert), seed=SEMILLA, reshuffle_each_iteration=True)
                           .batch(BATCH_SIZE))
    lotes_validacion = tf.data.Dataset.from_tensor_slices((x_validacion_finbert, y_validacion_finbert)).batch(BATCH_SIZE)
    historia = m4.fit(lotes_entrenamiento, validation_data=lotes_validacion, epochs=MAX_EPOCAS, callbacks=[parada])
    historial_m4 = {clave: [float(valor) for valor in valores] for clave, valores in historia.history.items()}
    historial_m4["minutos"] = round((time.time() - inicio) / 60, 1)
    m4.save_weights(RUTA_M4)
    RUTA_HISTORIAL_M4.write_text(json.dumps(historial_m4), encoding="utf-8")
    print(f"m4_finbert: {len(historial_m4['loss'])} épocas en {historial_m4['minutos']} min,"
          f" la mejor es la {int(np.argmin(historial_m4['val_loss'])) + 1}")
print(f"Parámetros: {m4.count_params():,}")

m4_finbert: ya ajustado, se lee de disco (4 épocas, la mejor la 1)
Parámetros: 109,754,115


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/saving/saving_lib.py:869: UserWarning:

Skipping variable loading for optimizer 'adam_4', because it has 2 variables whereas the saved optimizer has 404 variables. 



In [60]:
dibujar_perdidas(historial_m4, "M4 FinBERT (ajustado entero, una fase): pérdida por época")

En la validación, junto a M2 y al M3 oficial; el modelo tiene que quedarse con la pérdida mínima de su historial:

In [61]:
evaluacion_m4 = evaluar_clasificacion(reales_validacion, predecir(m4, x_validacion_finbert))
perdida_m4 = m4.evaluate(x_validacion_finbert, y_validacion_finbert, batch_size=BATCH_SIZE, verbose=0)[0]
print(f"M4: {len(historial_m4['val_loss'])} épocas, la mejor es la {int(np.argmin(historial_m4['val_loss'])) + 1}"
      f" · pérdida del modelo final: {perdida_m4:.4f} · mínima del historial: {min(historial_m4['val_loss']):.4f}"
      f" · última época: {historial_m4['val_loss'][-1]:.4f}")

pd.DataFrame({
    nombre: {"F1 macro": e["f1_macro"], "IC 95 % inferior": e["f1_ic_inferior"], "IC 95 % superior": e["f1_ic_superior"],
             "exactitud": e["exactitud"]}
    for nombre, e in [("M2 Transformer", evaluacion_m2), ("M3 BERT (corregido)", evaluacion_m3_corregido),
                      ("M4 FinBERT", evaluacion_m4)]
}).T.round(3)

M4: 4 épocas, la mejor es la 1 · pérdida del modelo final: 0.4052 · mínima del historial: 0.4052 · última época: 0.6560


,F1 macro,IC 95 % inferior,IC 95 % superior,exactitud
M2 Transformer,0.675,0.639,0.712,0.754
M3 BERT (corregido),0.600,0.561,0.640,0.714
M4 FinBERT,0.807,0.772,0.834,0.835


### 11.4 El examen de M4 con el test

Con M4 ya fijado, se mira su test **una sola vez**, junto a los demás modelos (M3, el corregido), con todas las frases y con las de AllAgree. Como referencia se repite M3 − M2.

In [62]:
predicciones_phrasebank["M4 FinBERT"] = (predecir(m4, preparar_finbert("validacion")[0])
                                         + predecir(m4, preparar_finbert("test")[0]))
predicciones_phrasebank.to_parquet(CARPETA_CACHE / "predicciones_phrasebank.parquet", index=False)
test = predicciones_phrasebank[predicciones_phrasebank["parte"] == "test"]

MODELOS_CON_M4 = ["siempre neutral", "M1 diccionario", "M2 Transformer", "M3 BERT (corregido)", "M4 FinBERT"]
resultados_m4, evaluaciones_m4_test = {}, {}
for nombre_examen, frases_examen in [("test", test), ("test, solo AllAgree", test[test["en_allagree"]])]:
    for modelo in MODELOS_CON_M4:
        e = evaluar_clasificacion(frases_examen["etiqueta"], frases_examen[modelo])
        evaluaciones_m4_test[(nombre_examen, modelo)] = e
        resultados_m4[(nombre_examen, modelo)] = {"F1 macro": e["f1_macro"], "IC 95 % inferior": e["f1_ic_inferior"],
                                                   "IC 95 % superior": e["f1_ic_superior"], "exactitud": e["exactitud"]}
pd.DataFrame(resultados_m4).T.round(3)

F1 macro  IC 95 % inferior  \
test                siempre neutral         0.248             0.240   
                    M1 diccionario          0.502             0.463   
                    M2 Transformer          0.673             0.632   
                    M3 BERT (corregido)     0.586             0.544   
                    M4 FinBERT              0.813             0.782   
test, solo AllAgree siempre neutral         0.250             0.239   
                    M1 diccionario          0.469             0.415   
                    M2 Transformer          0.807             0.758   
                    M3 BERT (corregido)     0.739             0.686   
                    M4 FinBERT              0.925             0.893   

                                         IC 95 % superior  exactitud  
test                siempre neutral                 0.256      0.594  
                    M1 diccionario                  0.539      0.631  
                    M2 Transformer                  0.709      0.746  
                    M3 BERT (corregido)             0.623      0.696  
                    M4 FinBERT                      0.841      0.844  
test, solo AllAgree siempre neutral                 0.262      0.600  
                    M1 diccionario                  0.520      0.638  
                    M2 Transformer                  0.850      0.864  
                    M3 BERT (corregido)             0.785      0.817  
                    M4 FinBERT                      0.953      0.946

In [63]:
PREGUNTAS_M4 = {
    "¿Partir de FinBERT supera a entrenar solo con PhraseBank? (M4 − M2)": ("M4 FinBERT", "M2 Transformer"),
    "¿Partir de FinBERT supera a preentrenar uno propio? (M4 − M3)": ("M4 FinBERT", "M3 BERT (corregido)"),
    "Referencia, la única limpia: ¿aporta preentrenar? (M3 − M2)": ("M3 BERT (corregido)", "M2 Transformer"),
}

diferencias_m4 = {}
for nombre_examen, frases_examen in [("test", test), ("test, solo AllAgree", test[test["en_allagree"]])]:
    for pregunta, (modelo_a, modelo_b) in PREGUNTAS_M4.items():
        d = comparar_f1(frases_examen["etiqueta"], frases_examen[modelo_a], frases_examen[modelo_b])
        diferencias_m4[(nombre_examen, pregunta)] = {
            "diferencia": round(d["diferencia"], 3), "IC 95 % inferior": round(d["ic_inferior"], 3),
            "IC 95 % superior": round(d["ic_superior"], 3),
            "¿concluyente?": "sí" if d["ic_inferior"] > 0 or d["ic_superior"] < 0 else "no (el 0 está dentro)"}
pd.DataFrame(diferencias_m4).T

diferencia  \
test                ¿Partir de FinBERT supera a entrenar solo con PhraseBank? (M4 − M2)       0.14   
                    ¿Partir de FinBERT supera a preentrenar uno propio? (M4 − M3)            0.226   
                    Referencia, la única limpia: ¿aporta preentrenar? (M3 − M2)             -0.087   
test, solo AllAgree ¿Partir de FinBERT supera a entrenar solo con PhraseBank? (M4 − M2)      0.118   
                    ¿Partir de FinBERT supera a preentrenar uno propio? (M4 − M3)            0.186   
                    Referencia, la única limpia: ¿aporta preentrenar? (M3 − M2)             -0.068   

                                                                                        IC 95 % inferior  \
test                ¿Partir de FinBERT supera a entrenar solo con PhraseBank? (M4 − M2)              0.1   
                    ¿Partir de FinBERT supera a preentrenar uno propio? (M4 − M3)                  0.186   
                    Referencia, la única limpia: ¿aporta preentrenar? (M3 − M2)                   -0.131   
test, solo AllAgree ¿Partir de FinBERT supera a entrenar solo con PhraseBank? (M4 − M2)            0.068   
                    ¿Partir de FinBERT supera a preentrenar uno propio? (M4 − M3)                  0.132   
                    Referencia, la única limpia: ¿aporta preentrenar? (M3 − M2)                   -0.124   

                                                                                        IC 95 % superior  \
test                ¿Partir de FinBERT supera a entrenar solo con PhraseBank? (M4 − M2)             0.18   
                    ¿Partir de FinBERT supera a preentrenar uno propio? (M4 − M3)                  0.273   
                    Referencia, la única limpia: ¿aporta preentrenar? (M3 − M2)                   -0.047   
test, solo AllAgree ¿Partir de FinBERT supera a entrenar solo con PhraseBank? (M4 − M2)            0.173   
                    ¿Partir de FinBERT supera a preentrenar uno propio? (M4 − M3)                  0.242   
                    Referencia, la única limpia: ¿aporta preentrenar? (M3 − M2)                   -0.018   

                                                                                        ¿concluyente?  
test                ¿Partir de FinBERT supera a entrenar solo con PhraseBank? (M4 − M2)            sí  
                    ¿Partir de FinBERT supera a preentrenar uno propio? (M4 − M3)                  sí  
                    Referencia, la única limpia: ¿aporta preentrenar? (M3 − M2)                    sí  
test, solo AllAgree ¿Partir de FinBERT supera a entrenar solo con PhraseBank? (M4 − M2)            sí  
                    ¿Partir de FinBERT supera a preentrenar uno propio? (M4 − M3)                  sí  
                    Referencia, la única limpia: ¿aporta preentrenar? (M3 − M2)                    sí

Dónde acierta y dónde falla M4 en el test (filas: la etiqueta real; columnas: lo que dice el modelo), junto a M2:

In [64]:
for modelo in ["M2 Transformer", "M4 FinBERT"]:
    print(modelo)
    display(evaluaciones_m4_test[("test", modelo)]["matriz_confusion"])

M2 Transformer


predicha,negative,neutral,positive
real,,,
negative,57,31,33
neutral,18,502,55
positive,5,104,163


M4 FinBERT


predicha,negative,neutral,positive
real,,,
negative,102,16,3
neutral,24,528,23
positive,15,70,187


**Conclusión de M4: partir de un BERT financiero preentrenado por otros es, con diferencia, la mejor receta de las cuatro.**

| Modelo | F1 macro en el test (IC 95 %) | Exactitud | Con AllAgree |
|---|---|---|---|
| Siempre neutral | 0,248 (0,24 – 0,26) | 0,594 | 0,250 |
| M1 diccionario | 0,502 (0,46 – 0,54) | 0,631 | 0,469 |
| M2 Transformer | 0,673 (0,63 – 0,71) | 0,746 | 0,807 |
| M3 BERT (corregido) | 0,586 (0,54 – 0,62) | 0,696 | 0,739 |
| **M4 FinBERT** | **0,813** (0,78 – 0,84) | **0,844** | **0,925** |

- **El test confirma la validación:** M4 saca 0,807 en la validación y 0,813 en el test. Como el resto de redes, se sobreajusta enseguida: la mejor época es la primera, y después la pérdida de validación sube de 0,41 a 0,66 mientras la de entrenamiento baja.
- **M4 − M2 = +0,140** (IC 95 %: 0,10 – 0,18) y **M4 − M3 = +0,226** (0,19 – 0,27). El 0 queda fuera en los dos casos, y también con AllAgree: +0,118 (0,07 – 0,17) y +0,186 (0,13 – 0,24).
- **Dónde gana: en las negativas**, la clase más difícil de todas. Acierta 102 de 121, frente a 57 de M2 y 34 de M3. En las positivas acierta 187 de 272 (M2, 163). Su error principal sigue siendo llamar neutrales a frases positivas: 70 de 272.
- **Cómo leerlo.** M4 − M2 y M4 − M3 comparan **recetas completas**: partir de un BERT de 110 M de parámetros preentrenado por otros con 4.900 M de palabras financieras, frente a entrenar desde cero o preentrenar uno propio pequeño. No dicen qué parte de la mejora viene del tamaño de la red, del corpus o del tokenizador, porque cambian a la vez. La única comparación limpia del preentrenamiento sigue siendo M3 − M2 (−0,087). Juntas cuentan una historia coherente: un preentrenamiento pequeño y propio no aporta, y partir de uno grande y financiero, sí.
- **Transparencia:** M4 se añadió después de mirar el test de M1–M3, por sugerencia del tutor. Sus decisiones se fijaron antes de entrenarlo y su test se miró una sola vez. FinBERT no vio nunca las etiquetas de PhraseBank: se preentrenó adivinando palabras tapadas en informes y conferencias, no clasificando sentimiento.

**Para el notebook 08:** M4 pasa a ser el mejor instrumento para medir el sentimiento. Los cuatro modelos puntúan los titulares, para ver si el que mejor clasifica encuentra más relación con la bolsa.

## Comprobaciones

In [65]:
# Lectura: las cifras de la ficha del dataset, tres etiquetas y ninguna frase vacía
assert len(frases_50) == 4846 and len(frases_todas) == 2264
for tabla in [frases_50, frases_todas]:
    assert set(tabla["etiqueta"]) == {"negative", "neutral", "positive"}
    assert (tabla["frase"].str.len() > 0).all()

# AllAgree es un subconjunto de 50Agree con las mismas etiquetas
pares_50 = set(zip(frases_50["frase"], frases_50["etiqueta"]))
assert all(par in pares_50 for par in zip(frases_todas["frase"], frases_todas["etiqueta"]))

# La normalización iguala frases que solo se diferencian en espacios y mayúsculas
assert normalizar("Gran , the company  GREW .") == normalizar("gran, the company grew.")

# Ninguna frase etiquetada está en el corpus de preentrenamiento
assert not en_el_corpus.any()
print("PhraseBank: OK")

PhraseBank: OK


In [66]:
# Partición: tamaños 60/20/20, sin frases repetidas dentro ni entre partes, y la misma proporción de clases
tamanos = particion["parte"].value_counts()
assert len(particion) == len(frases) == 4836
assert abs(tamanos["entrenamiento"] - 0.6 * len(particion)) <= 2
assert abs(tamanos["validacion"] - 0.2 * len(particion)) <= 2 and abs(tamanos["test"] - 0.2 * len(particion)) <= 2
assert not particion["frase"].map(normalizar).duplicated().any()
assert (proporciones[["negative", "neutral", "positive"]].max() - proporciones[["negative", "neutral", "positive"]].min()).max() < 1.0  # puntos %

# Las contradictorias no están, y en_allagree coincide con AllAgree
assert not particion["frase"].isin(contradictorias).any()
assert particion["en_allagree"].sum() == frases_todas["frase"].nunique()
print("Partición: OK")

Partición: OK


In [67]:
# Evaluación: coincide con sklearn en la referencia, el intervalo contiene el valor y la matriz cuenta todas las frases
assert abs(evaluacion_neutral["f1_macro"] - f1_score(reales, siempre_neutral, average="macro")) < 1e-12
assert abs(evaluacion_neutral["exactitud"] - accuracy_score(reales, siempre_neutral)) < 1e-12
assert evaluacion_neutral["f1_ic_inferior"] <= evaluacion_neutral["f1_macro"] <= evaluacion_neutral["f1_ic_superior"]
assert evaluacion_neutral["matriz_confusion"].values.sum() == len(reales)
assert evaluacion_neutral["matriz_confusion"]["neutral"].to_dict() == reales.value_counts().to_dict()

# Un modelo perfecto saca 1 en todo, con el intervalo [1, 1]
perfecto = evaluar_clasificacion(reales, reales)
assert perfecto["f1_macro"] == perfecto["f1_ic_inferior"] == perfecto["f1_ic_superior"] == perfecto["exactitud"] == 1

# Misma semilla, mismo intervalo; y una etiqueta desconocida da error en lugar de un resultado engañoso
assert evaluar_clasificacion(reales, siempre_neutral)["f1_ic_inferior"] == evaluacion_neutral["f1_ic_inferior"]
try:
    evaluar_clasificacion(["neutral"], ["neutro"])
    raise AssertionError("Debería haber fallado con una clase desconocida")
except ValueError:
    pass
print("Evaluación: OK")

Evaluación: OK


In [68]:
# Diccionario: el fichero esperado, una fila por palabra, sin palabras vacías, y las dos listas de tono sin solaparse
assert len(diccionario_lm) == 86553 and diccionario_lm["Word"].is_unique
assert (diccionario_lm["Word"].str.len() > 0).all() and (diccionario_lm["Word"] == diccionario_lm["Word"].str.upper()).all()
assert "NULL" in set(diccionario_lm["Word"])   # leída como palabra, no como dato vacío
assert len(palabras_negativas) == 2345 and len(palabras_positivas) == 347
assert not palabras_negativas & palabras_positivas
assert "BROKEN" not in palabras_negativas   # quitada en 2020 (valor negativo): no cuenta
print("Diccionario: OK")

Diccionario: OK


In [69]:
# M1: el tono está entre -1 y 1, el umbral elegido es el de mayor F1 (el menor si empatan) y el JSON guarda lo mismo
assert validacion["tono_m1"].between(-1, 1).all()
assert umbral_m1 in umbrales and f1_por_umbral[umbral_m1] == f1_por_umbral.max()
assert umbral_m1 == min(u for u in umbrales if f1_por_umbral[u] == f1_por_umbral.max())
assert abs(evaluacion_m1["f1_macro"] - f1_por_umbral.max()) < 1e-12
assert evaluacion_m1["f1_ic_inferior"] > 0.248   # M1 supera claramente a "siempre neutral"
guardado = json.loads(RUTA_M1.read_text(encoding="utf-8"))
assert guardado["umbral"] == umbral_m1 and set(guardado["palabras_negativas"]) == palabras_negativas
# Un ejemplo hecho a mano: 1 positiva y 2 negativas en 6 palabras (la coma y el punto no cuentan) -> tono -1/6
assert tono_loughran_mcdonald("Strong sales , but losses and declines .", {"LOSSES", "DECLINES"}, {"STRONG"}) == (1 - 2) / 6
print("M1: OK")

M1: OK


In [70]:
# Vocabulario: 30.000 palabras, [mask] la última, guardado y releído igual, y ninguna frase de PhraseBank cortada
assert len(vocabulario) == VOCAB_SIZE and vocabulario[:2] == ["", "[UNK]"]
assert mask_token_id == VOCAB_SIZE - 1 and vocabulario[mask_token_id] == "[mask]"
releido = TextVectorization(max_tokens=VOCAB_SIZE, output_mode="int", standardize=custom_standardization, output_sequence_length=MAX_LEN)
releido.set_vocabulary(RUTA_VOCABULARIO.read_text(encoding="utf-8").splitlines())
assert (releido(validacion["frase"].tolist()).numpy() == numeros_validacion).all()
assert longitudes["PhraseBank"].max() <= MAX_LEN
assert not corpus.str.contains("[mask]", case=False, regex=False).any()   # el token no aparece en ningún texto real
print("Vocabulario: OK")

Vocabulario: OK


In [71]:
# M2: la parada respeta la paciencia, el modelo guardado tiene los pesos de la mejor época y la elección es la de mayor F1
for historial in [historial_sin_pesos, historial_con_pesos]:
    epocas, mejor = len(historial["val_loss"]), int(np.argmin(historial["val_loss"])) + 1
    assert epocas == MAX_EPOCAS or epocas == mejor + PACIENCIA
assert np.isclose(m2.evaluate(x_validacion, y_validacion, verbose=0)[0], min(historial_m2["val_loss"]), atol=1e-4)
assert evaluacion_m2["f1_macro"] == max(e["f1_macro"] for e in evaluaciones_m2.values())
assert json.loads((CARPETA_MODELOS / "m2_eleccion.json").read_text(encoding="utf-8"))["version"] == version_m2

print("M2: OK")

M2: OK


In [72]:
# Preentrenamiento de M3: el enmascarado de los apuntes, 2 épocas con la pérdida bajando, y mejor que decir siempre la palabra más tapada
assert not tapadas[x_all_review <= 2].any()                                   # relleno, [UNK] y "the" nunca se tapan
assert abs(tapadas.sum() / (x_all_review > 2).sum() - 0.15) < 0.005           # el 15 % de las palabras que se pueden tapar
assert abs((x_masked_train[tapadas] == mask_token_id).mean() - 0.80) < 0.01   # 80 % con [mask]
assert abs((x_masked_train[tapadas] == x_all_review[tapadas]).mean() - 0.10) < 0.01   # 10 % sin cambiar
assert len(historial_mlm["loss"]) == EPOCAS_PREENTRENAMIENTO
assert historial_mlm["loss"][-1] < historial_mlm["loss"][0]
assert acierto_mlm > acierto_frecuente
print("Preentrenamiento de M3: OK")


Preentrenamiento de M3: OK


In [73]:
# M3: la fase 1 deja el encoder como salió del preentrenamiento, la fase 2 lo cambia, y las dos se quedan con su mejor época
pesos_preentrenados = mlm_model.get_layer("encoder_0").get_weights()
pesos_fase_1 = m3_congelado.get_layer("bert_preentrenado").get_layer("encoder_0").get_weights()
pesos_fase_2 = m3.get_layer("bert_preentrenado").get_layer("encoder_0").get_weights()
assert all(np.array_equal(antes, despues) for antes, despues in zip(pesos_preentrenados, pesos_fase_1))
assert not all(np.array_equal(antes, despues) for antes, despues in zip(pesos_preentrenados, pesos_fase_2))

for modelo, historial in [(m3_congelado, historial_congelado), (m3, historial_m3)]:
    epocas, mejor = len(historial["val_loss"]), int(np.argmin(historial["val_loss"])) + 1
    assert epocas == MAX_EPOCAS or epocas == mejor + PACIENCIA
    assert np.isclose(modelo.evaluate(x_validacion, y_validacion, verbose=0)[0], min(historial["val_loss"]), atol=1e-4)
print("M3: OK")

M3: OK


In [74]:
# Test: las 968 frases, mirado una vez; las predicciones guardadas dan en la validación lo mismo que los modelos elegidos
validacion_guardada = predicciones_phrasebank[predicciones_phrasebank["parte"] == "validacion"]
assert len(test) == 968 and test["en_allagree"].sum() == 448 and len(validacion_guardada) == 967
for modelo, evaluacion in [("M1 diccionario", evaluacion_m1), ("M2 Transformer", evaluacion_m2), ("M3 BERT", evaluacion_m3)]:
    assert np.isclose(evaluar_clasificacion(validacion_guardada["etiqueta"], validacion_guardada[modelo])["f1_macro"],
                      evaluacion["f1_macro"])
assert pd.read_parquet(CARPETA_CACHE / "predicciones_phrasebank.parquet").equals(predicciones_phrasebank)

# El intervalo de cada diferencia contiene su valor
for modelo_a, modelo_b in PREGUNTAS.values():
    d = comparar_f1(test["etiqueta"], test[modelo_a], test[modelo_b])
    assert d["ic_inferior"] <= d["diferencia"] <= d["ic_superior"]
print("Test: OK")

Test: OK


In [75]:
# Corrección de M3: la permutación mezcla las dos fuentes en toda la época, 2 épocas, la fase 1 no toca el encoder
# corregido, las dos fases con su mejor época, y las predicciones del M3 corregido guardadas
assert all(abs(porcentaje - 42) <= 2 for porcentaje in titulares_por_decimo(len(es_titular)))
assert len(historial_mlm_mezclado["loss"]) == EPOCAS_PREENTRENAMIENTO
pesos_corregido = mlm_mezclado.get_layer("encoder_0").get_weights()
pesos_corregido_fase_1 = m3_corregido_congelado.get_layer("bert_preentrenado").get_layer("encoder_0").get_weights()
assert all(np.array_equal(antes, despues) for antes, despues in zip(pesos_corregido, pesos_corregido_fase_1))
for modelo, historial in [(m3_corregido_congelado, historial_corregido_congelado), (m3_corregido, historial_m3_corregido)]:
    epocas, mejor = len(historial["val_loss"]), int(np.argmin(historial["val_loss"])) + 1
    assert epocas == MAX_EPOCAS or epocas == mejor + PACIENCIA
    assert np.isclose(modelo.evaluate(x_validacion, y_validacion, verbose=0)[0], min(historial["val_loss"]), atol=1e-4)
guardadas = pd.read_parquet(CARPETA_CACHE / "predicciones_phrasebank.parquet")
assert guardadas["M3 BERT (corregido)"].isin(CLASES_SENTIMIENTO).all()
assert np.isclose(evaluar_clasificacion(guardadas.loc[guardadas["parte"] == "validacion", "etiqueta"],
                                        guardadas.loc[guardadas["parte"] == "validacion", "M3 BERT (corregido)"])["f1_macro"],
                  evaluacion_m3_corregido["f1_macro"])
print("Corrección de M3: OK")

Corrección de M3: OK


In [76]:
# M4: FinBERT convertido con su vocabulario, ninguna frase cortada, la misma partición, la GELU exacta en las 12 capas,
# la parada con paciencia y la mejor época, el resultado no depende del relleno y las predicciones guardadas
configuracion_finbert = json.loads((CARPETA_FINBERT / "config.json").read_text(encoding="utf-8"))
assert configuracion_finbert["model_type"] == "bert"
assert configuracion_finbert["vocab_size"] == tokenizador_finbert.vocabulary_size() == 30873
assert tokens_por_frase.max() <= SECUENCIA_FINBERT
assert (y_entrenamiento_finbert == y_entrenamiento).all() and (y_validacion_finbert == y_validacion).all()
assert len(m4.backbone.transformer_layers) == 12
assert all(capa._feedforward_intermediate_dense.activation is keras.activations.gelu for capa in m4.backbone.transformer_layers)

epocas, mejor = len(historial_m4["val_loss"]), int(np.argmin(historial_m4["val_loss"])) + 1
assert epocas == MAX_EPOCAS or epocas == mejor + PACIENCIA
assert np.isclose(perdida_m4, min(historial_m4["val_loss"]), atol=1e-4)

# Con la cabeza [CLS], rellenar hasta 64 o hasta 136 tokens da las mismas probabilidades (lo que usa el notebook 08)
frases_prueba = particion["frase"].iloc[:16].tolist()
preprocesador_corto = keras_hub.models.BertTextClassifierPreprocessor(tokenizador_finbert, sequence_length=64)
assert np.allclose(m4.predict(preprocesador_corto(frases_prueba), verbose=0),
                   m4.predict(preprocesador_finbert(frases_prueba), verbose=0), atol=1e-4)

guardadas = pd.read_parquet(CARPETA_CACHE / "predicciones_phrasebank.parquet")
assert guardadas["M4 FinBERT"].isin(CLASES_SENTIMIENTO).all() and len(guardadas) == 967 + 968
assert np.isclose(evaluar_clasificacion(guardadas.loc[guardadas["parte"] == "validacion", "etiqueta"],
                                        guardadas.loc[guardadas["parte"] == "validacion", "M4 FinBERT"])["f1_macro"],
                  evaluacion_m4["f1_macro"])
print("M4: OK")

M4: OK
